# County-Year Vulnerability Dataset Building

## 1. Revised Dataset Setup

Prepare the existing Zillow-FEMA panel for the revised county-year vulnerability framework.

### 1.1 Imports and Project Paths

In [48]:
# ==================================================
# Imports
# ==================================================

from pathlib import Path

import pandas as pd
import numpy as np

# ==================================================
# Project Paths
# ==================================================

PROJECT_ROOT = Path("..")

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "merged_zillow_fema_data.csv"
)

RAW_DATA = (
    PROJECT_ROOT
    / "data"
    / "raw"
)

PROCESSED_DATA = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

INTERIM_DATA = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "county_year_features"
)


RESULTS_TABLES = (
    PROJECT_ROOT
    / "results"
    / "tables"
)

RESULTS_LOGS = (
    PROJECT_ROOT
    / "results"
    / "logs"
)


# ==================================================
# Project Settings
# ==================================================

PROJECT_START_YEAR = 2011
PROJECT_END_YEAR = 2025

### 1.2 Load Existing Monthly Dataset

In [2]:
# ==================================================
# Load Existing Cleaned Zillow-FEMA Dataset
# ==================================================

df = pd.read_csv(DATA_PATH)

# convert date column to datetime
df["Date"] = pd.to_datetime(df["Date"])

print("Dataset shape:", df.shape)
print("Date range:", df["Date"].min(), "to", df["Date"].max())
print("Number of counties:", df["STCOFIPS"].nunique())

df.head()

Dataset shape: (12767, 18)
Date range: 2010-01-31 00:00:00 to 2025-12-31 00:00:00
Number of counties: 67


,RegionID,SizeRank,RegionName,State,Metro,StateCodeFIPS,MunicipalCodeFIPS,Date,HousingPrice,STCOFIPS,STATE,STATEABBRV,COUNTY,POPULATION,CFLD_RISKS,HRCN_RISKS,SOVI_SCORE,RESL_SCORE
0,67,371,Bay County,FL,"Panama City, FL",12,5,2010-01-31,177323.256436,12005,Florida,FL,Bay,174869,77.2,98.664998,30.852417,29.452926
1,67,371,Bay County,FL,"Panama City, FL",12,5,2010-02-28,175726.799272,12005,Florida,FL,Bay,174869,77.2,98.664998,30.852417,29.452926
2,67,371,Bay County,FL,"Panama City, FL",12,5,2010-03-31,173648.913459,12005,Florida,FL,Bay,174869,77.2,98.664998,30.852417,29.452926
3,67,371,Bay County,FL,"Panama City, FL",12,5,2010-04-30,172490.101014,12005,Florida,FL,Bay,174869,77.2,98.664998,30.852417,29.452926
4,67,371,Bay County,FL,"Panama City, FL",12,5,2010-05-31,171749.827482,12005,Florida,FL,Bay,174869,77.2,98.664998,30.852417,29.452926


### 1.3 Dataset Validation

In [3]:
# ==================================================
# Dataset Validation
# ==================================================

num_rows = df.shape[0]
num_columns = df.shape[1]
num_counties = df["STCOFIPS"].nunique()
start_date = df["Date"].min()
end_date = df["Date"].max()

# duplicate county-month rows
duplicate_count = df.duplicated(
    subset=["STCOFIPS", "Date"]
).sum()

print("Rows:", num_rows)
print("Columns:", num_columns)
print("Counties:", num_counties)
print("Start Date:", start_date)
print("End Date:", end_date)
print("Duplicate county-month rows:", duplicate_count)

Rows: 12767
Columns: 18
Counties: 67
Start Date: 2010-01-31 00:00:00
End Date: 2025-12-31 00:00:00
Duplicate county-month rows: 0


In [4]:
# ==================================================
# Missing Value Check
# ==================================================

validation_cols = [
    "HousingPrice",
    "CFLD_RISKS",
    "HRCN_RISKS",
    "SOVI_SCORE",
    "RESL_SCORE",
    "POPULATION"
]

print("Missing Values:")
print(df[validation_cols].isnull().sum())

# ==================================================
# Negative Value Check
# ==================================================

numeric_checks = [
    "HousingPrice",
    "POPULATION"
]

print("\nNegative Values:")

for col in numeric_checks:
    negative_count = (df[col] < 0).sum()
    print(f"{col}: {negative_count}")

Missing Values:
HousingPrice    0
CFLD_RISKS      0
HRCN_RISKS      0
SOVI_SCORE      0
RESL_SCORE      0
POPULATION      0
dtype: int64

Negative Values:
HousingPrice: 0
POPULATION: 0


### 1.4 Key Takeaways

- The existing monthly Zillow-FEMA dataset is used as the input for the revised framework.
- The goal is to create annual housing-market indicators, not a direct price-prediction target.
- The county-year dataset will become the housing-market signal layer for vulnerability modeling.

## 2. County-Year Housing Market Indicators

Aggregate monthly Zillow values into annual county-level housing market signals.

### 2.1 Create Year Variable

In [5]:
# ==================================================
# Create Year Variable
# ==================================================

df["Year"] = df["Date"].dt.year

print("Years available:", df["Year"].min(), "to", df["Year"].max())
print("County-year combinations:", df[["STCOFIPS", "Year"]].drop_duplicates().shape[0])

Years available: 2010 to 2025
County-year combinations: 1064


### 2.2 Annual Housing Indicator Aggregation

In [7]:
# ==================================================
# County-Year Housing Indicator Aggregation
# ==================================================

# Handle missing Metro values consistently
df["Metro"] = df["Metro"].fillna("Non-Metro")

# Create metro indicator
df["is_metro"] = (df["Metro"] != "Non-Metro").astype(int)

id_cols = [
    "STCOFIPS",
    "RegionID",
    "RegionName",
    "State",
    "Metro",
    "StateCodeFIPS",
    "MunicipalCodeFIPS",
    "COUNTY"
]

county_year_housing = (
    df
    .groupby(id_cols + ["Year"], as_index=False)
    .agg(
        avg_annual_housing_price=("HousingPrice", "mean"),
        median_annual_housing_price=("HousingPrice", "median"),
        min_annual_housing_price=("HousingPrice", "min"),
        max_annual_housing_price=("HousingPrice", "max"),
        annual_price_volatility=("HousingPrice", "std"),
        monthly_observations=("HousingPrice", "count"),
        is_metro=("is_metro", "first"),
        SizeRank=("SizeRank", "first"),
        POPULATION=("POPULATION", "first"),
        CFLD_RISKS=("CFLD_RISKS", "first"),
        HRCN_RISKS=("HRCN_RISKS", "first"),
        SOVI_SCORE=("SOVI_SCORE", "first"),
        RESL_SCORE=("RESL_SCORE", "first")
    )
)

# Fill volatility for county-years with only one monthly observation
county_year_housing["annual_price_volatility"] = (
    county_year_housing["annual_price_volatility"]
    .fillna(0)
)

print("County-year housing shape:", county_year_housing.shape)
print("Counties:", county_year_housing["STCOFIPS"].nunique())
print("Years:", county_year_housing["Year"].min(), "to", county_year_housing["Year"].max())

county_year_housing.head()

County-year housing shape: (1064, 22)
Counties: 67
Years: 2010 to 2025


,STCOFIPS,RegionID,RegionName,State,Metro,StateCodeFIPS,MunicipalCodeFIPS,COUNTY,Year,avg_annual_housing_price,...,max_annual_housing_price,annual_price_volatility,monthly_observations,is_metro,SizeRank,POPULATION,CFLD_RISKS,HRCN_RISKS,SOVI_SCORE,RESL_SCORE
0,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2010,162126.833191,...,168775.803697,5833.729164,12,1,251,277984,0.0,96.704214,34.764631,80.979644
1,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2011,144811.606954,...,151065.304144,4024.282542,12,1,251,277984,0.0,96.704214,34.764631,80.979644
2,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2012,136832.426441,...,139626.338658,1809.886935,12,1,251,277984,0.0,96.704214,34.764631,80.979644
3,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2013,140015.149500,...,141791.385842,1500.697759,12,1,251,277984,0.0,96.704214,34.764631,80.979644
4,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2014,147354.521716,...,149219.273913,2125.835860,12,1,251,277984,0.0,96.704214,34.764631,80.979644


### 2.3 Annual Growth and Appreciation Features

In [8]:
# ==================================================
# Annual Growth and Appreciation Features
# ==================================================

county_year_housing = county_year_housing.sort_values(
    ["STCOFIPS", "Year"]
).reset_index(drop=True)

# previous-year average price
county_year_housing["prev_year_housing_price"] = (
    county_year_housing
    .groupby("STCOFIPS")["avg_annual_housing_price"]
    .shift(1)
)

# annual dollar growth
county_year_housing["annual_price_growth_dollar"] = (
    county_year_housing["avg_annual_housing_price"]
    - county_year_housing["prev_year_housing_price"]
)

# annual percentage growth
county_year_housing["annual_price_growth_pct"] = (
    county_year_housing["annual_price_growth_dollar"]
    / county_year_housing["prev_year_housing_price"]
    * 100
)

# baseline price for each county
county_year_housing["baseline_housing_price"] = (
    county_year_housing
    .groupby("STCOFIPS")["avg_annual_housing_price"]
    .transform("first")
)

# appreciation relative to first available year
county_year_housing["appreciation_from_baseline_pct"] = (
    (county_year_housing["avg_annual_housing_price"]
     - county_year_housing["baseline_housing_price"])
    / county_year_housing["baseline_housing_price"]
    * 100
)

# growth acceleration compared with previous annual growth rate
county_year_housing["price_growth_acceleration"] = (
    county_year_housing
    .groupby("STCOFIPS")["annual_price_growth_pct"]
    .diff()
)

county_year_housing.head()

,STCOFIPS,RegionID,RegionName,State,Metro,StateCodeFIPS,MunicipalCodeFIPS,COUNTY,Year,avg_annual_housing_price,...,CFLD_RISKS,HRCN_RISKS,SOVI_SCORE,RESL_SCORE,prev_year_housing_price,annual_price_growth_dollar,annual_price_growth_pct,baseline_housing_price,appreciation_from_baseline_pct,price_growth_acceleration
0,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2010,162126.833191,...,0.0,96.704214,34.764631,80.979644,NaN,NaN,NaN,162126.833191,0.000000,NaN
1,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2011,144811.606954,...,0.0,96.704214,34.764631,80.979644,162126.833191,-17315.226237,-10.680050,162126.833191,-10.680050,NaN
2,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2012,136832.426441,...,0.0,96.704214,34.764631,80.979644,144811.606954,-7979.180513,-5.510042,162126.833191,-15.601616,5.170008
3,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2013,140015.149500,...,0.0,96.704214,34.764631,80.979644,136832.426441,3182.723060,2.326001,162126.833191,-13.638510,7.836043
4,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2014,147354.521716,...,0.0,96.704214,34.764631,80.979644,140015.149500,7339.372216,5.241842,162126.833191,-9.111577,2.915841


### 2.4 Market Signal Flags

In [11]:
# ==================================================
# 2.4 Market Signal Flags
# ==================================================

# High annual growth threshold using valid growth values only
high_growth_threshold = county_year_housing["annual_price_growth_pct"].median()

# Create high-growth flag
# First-year rows have NaN growth, so keep the flag as NaN instead of forcing 0
county_year_housing["high_growth_flag"] = np.where(
    county_year_housing["annual_price_growth_pct"].isna(),
    np.nan,
    np.where(
        county_year_housing["annual_price_growth_pct"] >= high_growth_threshold,
        1,
        0
    )
)

# High volatility threshold using all county-years
high_volatility_threshold = county_year_housing["annual_price_volatility"].median()

# Create high-volatility flag
county_year_housing["high_volatility_flag"] = np.where(
    county_year_housing["annual_price_volatility"] >= high_volatility_threshold,
    1,
    0
)

# Complete-year flag based on monthly observations
county_year_housing["complete_year_flag"] = np.where(
    county_year_housing["monthly_observations"] == 12,
    1,
    0
)

print("High growth threshold:", round(high_growth_threshold, 2))
print("High volatility threshold:", round(high_volatility_threshold, 2))

county_year_housing[
    [
        "RegionName",
        "Year",
        "avg_annual_housing_price",
        "annual_price_growth_pct",
        "annual_price_volatility",
        "high_growth_flag",
        "high_volatility_flag",
        "complete_year_flag"
    ]
].head()

High growth threshold: 5.92
High volatility threshold: 3724.47


,RegionName,Year,avg_annual_housing_price,annual_price_growth_pct,annual_price_volatility,high_growth_flag,high_volatility_flag,complete_year_flag
0,Alachua County,2010,162126.833191,NaN,5833.729164,NaN,1,1
1,Alachua County,2011,144811.606954,-10.680050,4024.282542,0.0,1,1
2,Alachua County,2012,136832.426441,-5.510042,1809.886935,0.0,0,1
3,Alachua County,2013,140015.149500,2.326001,1500.697759,0.0,0,1
4,Alachua County,2014,147354.521716,5.241842,2125.835860,0.0,0,1


Section 2.5 — County-Year Housing Dataset Validation.

In [12]:
# ==================================================
# 2.5 County-Year Housing Dataset Validation
# ==================================================

print("County-year dataset shape:", county_year_housing.shape)
print("Unique counties:", county_year_housing["STCOFIPS"].nunique())
print("Year range:", county_year_housing["Year"].min(), "to", county_year_housing["Year"].max())

print("\nMonthly observation coverage:")
print(county_year_housing["monthly_observations"].value_counts().sort_index())

print("\nComplete-year coverage:")
print(county_year_housing["complete_year_flag"].value_counts())

print("\nMissing values:")
missing_summary = (
    county_year_housing
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_summary[missing_summary > 0]

County-year dataset shape: (1064, 31)
Unique counties: 67
Year range: 2010 to 2025

Monthly observation coverage:
monthly_observations
11       1
12    1063
Name: count, dtype: int64

Complete-year coverage:
complete_year_flag
1    1063
0       1
Name: count, dtype: int64

Missing values:


price_growth_acceleration     134
prev_year_housing_price        67
annual_price_growth_dollar     67
high_growth_flag               67
annual_price_growth_pct        67
dtype: int64

The county-year housing dataset contains all 67 Florida counties from 2010–2025. Monthly coverage is nearly complete, with only one county-year containing 11 observations instead of 12. Missing values appear only in lagged growth and acceleration features, which is expected because the first year has no previous-year comparison and acceleration requires two years of growth history.

## 3. Final County-Year Housing Layer

Filter the annual dataset to the project period and validate the housing-market signal layer.

### 3.1 Filter to Project Period

In [13]:
# ==================================================
# Filter to Project Period
# ==================================================

county_year_housing_final = county_year_housing[
    (county_year_housing["Year"] >= PROJECT_START_YEAR)
    & (county_year_housing["Year"] <= PROJECT_END_YEAR)
].copy()

print("Final county-year housing shape:", county_year_housing_final.shape)
print("Counties:", county_year_housing_final["STCOFIPS"].nunique())
print("Years:", county_year_housing_final["Year"].min(), "to", county_year_housing_final["Year"].max())
print("Duplicate county-year rows:", county_year_housing_final.duplicated(subset=["STCOFIPS", "Year"]).sum())

county_year_housing_final.head()

Final county-year housing shape: (1000, 31)
Counties: 67
Years: 2011 to 2025
Duplicate county-year rows: 0


,STCOFIPS,RegionID,RegionName,State,Metro,StateCodeFIPS,MunicipalCodeFIPS,COUNTY,Year,avg_annual_housing_price,...,RESL_SCORE,prev_year_housing_price,annual_price_growth_dollar,annual_price_growth_pct,baseline_housing_price,appreciation_from_baseline_pct,price_growth_acceleration,high_growth_flag,high_volatility_flag,complete_year_flag
1,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2011,144811.606954,...,80.979644,162126.833191,-17315.226237,-10.680050,162126.833191,-10.680050,NaN,0.0,1,1
2,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2012,136832.426441,...,80.979644,144811.606954,-7979.180513,-5.510042,162126.833191,-15.601616,5.170008,0.0,0,1
3,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2013,140015.149500,...,80.979644,136832.426441,3182.723060,2.326001,162126.833191,-13.638510,7.836043,0.0,0,1
4,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2014,147354.521716,...,80.979644,140015.149500,7339.372216,5.241842,162126.833191,-9.111577,2.915841,0.0,0,1
5,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2015,153750.122064,...,80.979644,147354.521716,6395.600348,4.340281,162126.833191,-5.166764,-0.901560,0.0,0,1


### 3.2 County-Year Dataset Overview

In [14]:
# ==================================================
# County-Year Dataset Overview Table
# ==================================================

county_year_overview = pd.DataFrame({
    "Metric": [
        "Observations",
        "Variables",
        "Counties",
        "Start Year",
        "End Year",
        "Duplicate County-Year Rows",
        "Complete County-Year Rows"
    ],
    "Value": [
        county_year_housing_final.shape[0],
        county_year_housing_final.shape[1],
        county_year_housing_final["STCOFIPS"].nunique(),
        county_year_housing_final["Year"].min(),
        county_year_housing_final["Year"].max(),
        county_year_housing_final.duplicated(subset=["STCOFIPS", "Year"]).sum(),
        int(county_year_housing_final["complete_year_flag"].sum())
    ]
})

# display table
display(county_year_overview)

# save table
county_year_overview.to_csv(
    RESULTS_TABLES / "county_year_housing_overview.csv",
    index=False
)

print(
    f"Saved: "
    f"{RESULTS_TABLES / 'county_year_housing_overview.csv'}"
)

,Metric,Value
0,Observations,1000
1,Variables,31
2,Counties,67
3,Start Year,2011
4,End Year,2025
5,Duplicate County-Year Rows,0
6,Complete County-Year Rows,999


Saved: ..\results\tables\county_year_housing_overview.csv


### 3.3 Annual Housing Summary

In [15]:
# ==================================================
# Annual Housing Summary Table
# ==================================================

annual_housing_summary = (
    county_year_housing_final
    .groupby("Year", as_index=False)
    .agg(
        avg_florida_housing_price=("avg_annual_housing_price", "mean"),
        median_florida_housing_price=("avg_annual_housing_price", "median"),
        avg_annual_growth_pct=("annual_price_growth_pct", "mean"),
        avg_annual_volatility=("annual_price_volatility", "mean"),
        county_count=("STCOFIPS", "nunique")
    )
    .round(2)
)

# display table
display(annual_housing_summary.head())

# save table
annual_housing_summary.to_csv(
    RESULTS_TABLES / "annual_housing_summary.csv",
    index=False
)

print(
    f"Saved: "
    f"{RESULTS_TABLES / 'annual_housing_summary.csv'}"
)

,Year,avg_florida_housing_price,median_florida_housing_price,avg_annual_growth_pct,avg_annual_volatility,county_count
0,2011,123820.62,116077.90,-7.58,2628.04,66
1,2012,122216.85,114642.82,-1.33,1760.46,66
2,2013,131230.99,125456.69,6.78,4036.37,66
3,2014,142503.33,134473.35,7.83,2901.59,66
4,2015,152479.77,145364.55,6.58,4011.38,66


Saved: ..\results\tables\annual_housing_summary.csv


### 3.4 Missing Value Check

In [16]:
# ==================================================
# Final Missing Value Check
# ==================================================

final_check_cols = [
    "avg_annual_housing_price",
    "annual_price_growth_pct",
    "annual_price_volatility",
    "appreciation_from_baseline_pct",
    "CFLD_RISKS",
    "HRCN_RISKS",
    "SOVI_SCORE",
    "RESL_SCORE",
    "POPULATION"
]

missing_summary = (
    county_year_housing_final[final_check_cols]
    .isnull()
    .sum()
    .reset_index()
    .rename(columns={"index": "Variable", 0: "Missing Values"})
)

missing_summary

,Variable,Missing Values
0,avg_annual_housing_price,0
1,annual_price_growth_pct,3
2,annual_price_volatility,0
3,appreciation_from_baseline_pct,0
4,CFLD_RISKS,0
5,HRCN_RISKS,0
6,SOVI_SCORE,0
7,RESL_SCORE,0
8,POPULATION,0


### 3.5 Save County-Year Housing Layer

In [ ]:
# ==================================================
# Save County-Year Housing Layer
# ==================================================

output_path = (
    INTERIM_DATA
    / "county_year_housing_indicators.csv"
)

county_year_housing_final.to_csv(
    output_path,
    index=False
)

print(f"Saved: {output_path}")
print("Final shape:", county_year_housing_final.shape)

Saved: ..\data\processed\county_year_housing_indicators.csv
Final shape: (1000, 31)


### Key Takeaways

- Monthly Zillow housing values were converted into county-year housing-market indicators.
- Housing price is now treated as a market signal layer rather than the only prediction target.
- The saved dataset will be used next for spatial features, disaster history, and socioeconomic indicators.

## 4. Next Step

Prepare county adjacency and neighboring-county housing features using the Florida county shapefile.

## 4. Spatial Feature Preparation

### 4.1 Load Florida County Shapefile

Load the Florida county boundary shapefile so spatial relationships between counties can be created.

In [23]:
# ==================================================
# 4.1 Load Florida County Shapefile
# ==================================================

import geopandas as gpd

# path to county shapefile
county_shapefile_path = (
    RAW_DATA
    / "shapefiles"
    / "tl_2025_us_county"
    / "tl_2025_us_county.shp"
)

# load county shapefile
counties_gdf = gpd.read_file(county_shapefile_path)

print("Original shapefile shape:", counties_gdf.shape)
print("Original CRS:", counties_gdf.crs)

counties_gdf.head()

Original shapefile shape: (3235, 19)
Original CRS: EPSG:4269


,STATEFP,COUNTYFP,COUNTYNS,GEOID,GEOIDFQ,NAME,NAMELSAD,LSAD,CLASSFP,MTFCC,CSAFP,CBSAFP,METDIVFP,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry
0,40,075,01101825,40075,0500000US40075,Kiowa,Kiowa County,06,H1,G4020,NaN,NaN,NaN,A,2629039892,40296743,+34.9214893,-098.9816168,"POLYGON ((-98.95506 35.11643, -98.94903 35.116..."
1,46,079,01265776,46079,0500000US46079,Lake,Lake County,06,H1,G4020,NaN,NaN,NaN,A,1457916151,31746795,+44.0284497,-097.1232229,"POLYGON ((-96.88886 43.9353, -96.88886 43.9351..."
2,37,033,01008542,37033,0500000US37033,Caswell,Caswell County,06,H1,G4020,NaN,NaN,NaN,A,1102042927,8293623,+36.3943252,-079.3396193,"POLYGON ((-79.14343 36.4422, -79.14345 36.4418..."
3,48,377,01383974,48377,0500000US48377,Presidio,Presidio County,06,H1,G4020,NaN,NaN,NaN,A,9985057447,1773188,+30.0058912,-104.2616192,"POLYGON ((-104.98078 30.62552, -104.98073 30.6..."
4,39,057,01074041,39057,0500000US39057,Greene,Greene County,06,H1,G4020,212,19430,NaN,A,1071302625,6798109,+39.6874785,-083.8948943,"POLYGON ((-84.10668 39.68891, -84.10662 39.689..."


In [24]:
# ==================================================
# Filter to Florida Counties
# ==================================================

florida_counties_gdf = counties_gdf[
    counties_gdf["STATEFP"] == "12"
].copy()

print("Florida county shapefile shape:", florida_counties_gdf.shape)
print("Florida counties:", florida_counties_gdf["COUNTYFP"].nunique())

florida_counties_gdf[
    ["STATEFP", "COUNTYFP", "GEOID", "NAME", "NAMELSAD", "geometry"]
].head()

Florida county shapefile shape: (67, 19)
Florida counties: 67


,STATEFP,COUNTYFP,GEOID,NAME,NAMELSAD,geometry
22,12,045,12045,Gulf,Gulf County,"POLYGON ((-85.39126 30.02816, -85.39123 30.028..."
46,12,105,12105,Polk,Polk County,"POLYGON ((-81.42455 28.01248, -81.42455 28.012..."
93,12,079,12079,Madison,Madison County,"POLYGON ((-83.82191 30.30749, -83.8219 30.3076..."
173,12,029,12029,Dixie,Dixie County,"MULTIPOLYGON (((-82.91871 29.82408, -82.91672 ..."
187,12,097,12097,Osceola,Osceola County,"POLYGON ((-81.65727 28.3471, -81.65666 28.3471..."


### 4.2 Validate County Coverage

Check whether the Florida shapefile counties match the county identifiers in the county-year housing dataset.

In [25]:
# ==================================================
# 4.2 Validate County Coverage
# ==================================================

# Make shapefile county ID consistent with housing dataset county ID
florida_counties_gdf["STCOFIPS"] = florida_counties_gdf["GEOID"].astype(str)

county_year_housing_final["STCOFIPS"] = (
    county_year_housing_final["STCOFIPS"]
    .astype(str)
    .str.zfill(5)
)

# unique county IDs
shapefile_counties = set(florida_counties_gdf["STCOFIPS"].unique())
housing_counties = set(county_year_housing_final["STCOFIPS"].unique())

# compare coverage
missing_in_shapefile = sorted(housing_counties - shapefile_counties)
missing_in_housing = sorted(shapefile_counties - housing_counties)

print("Counties in shapefile:", len(shapefile_counties))
print("Counties in housing dataset:", len(housing_counties))
print("Missing in shapefile:", missing_in_shapefile)
print("Missing in housing dataset:", missing_in_housing)

Counties in shapefile: 67
Counties in housing dataset: 67
Missing in shapefile: []
Missing in housing dataset: []


In [26]:
# ==================================================
# Matched County Name Check
# ==================================================

county_match_check = (
    florida_counties_gdf[["STCOFIPS", "NAME", "NAMELSAD"]]
    .merge(
        county_year_housing_final[["STCOFIPS", "RegionName", "COUNTY"]]
        .drop_duplicates(),
        on="STCOFIPS",
        how="inner"
    )
    .sort_values("STCOFIPS")
)

print("Matched counties:", county_match_check["STCOFIPS"].nunique())

county_match_check.head()

Matched counties: 67


,STCOFIPS,NAME,NAMELSAD,RegionName,COUNTY
63,12001,Alachua,Alachua County,Alachua County,Alachua
22,12003,Baker,Baker County,Baker County,Baker
8,12005,Bay,Bay County,Bay County,Bay
42,12007,Bradford,Bradford County,Bradford County,Bradford
32,12009,Brevard,Brevard County,Brevard County,Brevard


### 4.3 Create County Adjacency Table

Identify neighboring Florida counties using shared county boundaries.

In [27]:
# ==================================================
# 4.3 Create County Adjacency Table
# ==================================================

# keep only needed columns
county_geometries = florida_counties_gdf[
    ["STCOFIPS", "NAME", "NAMELSAD", "geometry"]
].copy()

# create spatial join to identify touching counties
county_neighbors = gpd.sjoin(
    county_geometries,
    county_geometries,
    how="inner",
    predicate="touches",
    lsuffix="county",
    rsuffix="neighbor"
)

# remove self-matches if any
county_neighbors = county_neighbors[
    county_neighbors["STCOFIPS_county"] != county_neighbors["STCOFIPS_neighbor"]
].copy()

# clean adjacency table
county_adjacency = county_neighbors[
    [
        "STCOFIPS_county",
        "NAME_county",
        "STCOFIPS_neighbor",
        "NAME_neighbor"
    ]
].rename(
    columns={
        "STCOFIPS_county": "county_fips",
        "NAME_county": "county_name",
        "STCOFIPS_neighbor": "neighbor_fips",
        "NAME_neighbor": "neighbor_name"
    }
).sort_values(
    ["county_fips", "neighbor_fips"]
).reset_index(drop=True)

print("County adjacency rows:", county_adjacency.shape[0])
print("Counties with neighbors:", county_adjacency["county_fips"].nunique())

county_adjacency.head(10)

County adjacency rows: 320
Counties with neighbors: 67


,county_fips,county_name,neighbor_fips,neighbor_name
0,12001,Alachua,12007,Bradford
1,12001,Alachua,12023,Columbia
2,12001,Alachua,12041,Gilchrist
3,12001,Alachua,12075,Levy
4,12001,Alachua,12083,Marion
5,12001,Alachua,12107,Putnam
6,12001,Alachua,12125,Union
7,12003,Baker,12007,Bradford
8,12003,Baker,12019,Clay
9,12003,Baker,12023,Columbia


In [28]:
# ==================================================
# Neighbor Count Validation
# ==================================================

neighbor_counts = (
    county_adjacency
    .groupby(["county_fips", "county_name"], as_index=False)
    .agg(neighbor_count=("neighbor_fips", "nunique"))
    .sort_values("neighbor_count")
)

print("Minimum neighbors:", neighbor_counts["neighbor_count"].min())
print("Maximum neighbors:", neighbor_counts["neighbor_count"].max())
print("Average neighbors:", round(neighbor_counts["neighbor_count"].mean(), 2))

neighbor_counts.head(10)

Minimum neighbors: 1
Maximum neighbors: 10
Average neighbors: 4.78


,county_fips,county_name,neighbor_count
15,12033,Escambia,1
43,12087,Monroe,2
45,12091,Okaloosa,2
56,12113,Santa Rosa,2
51,12103,Pinellas,2
44,12089,Nassau,2
17,12037,Franklin,3
22,12047,Hamilton,3
42,12086,Miami-Dade,3
55,12111,St. Lucie,3


In [ ]:
# ==================================================
# Save County Adjacency Table
# ==================================================

county_adjacency.to_csv(
    INTERIM_DATA / "florida_county_adjacency.csv",
    index=False
)

print(
    f"Saved: "
    f"{INTERIM_DATA / 'florida_county_adjacency.csv'}"
)

Saved: ..\data\processed\florida_county_adjacency.csv


### 4.4 Create Neighboring County Housing Features

Use the county adjacency table to calculate neighboring-county housing market indicators for each county-year.

In [30]:
# ==================================================
# 4.4 Create Neighboring County Housing Features
# ==================================================

# Keep housing variables needed for neighboring-county features
neighbor_housing_base = county_year_housing_final[
    [
        "STCOFIPS",
        "Year",
        "avg_annual_housing_price",
        "annual_price_growth_pct",
        "annual_price_volatility",
        "high_growth_flag",
        "high_volatility_flag"
    ]
].copy()

# Rename columns so they represent neighbor values
neighbor_housing_base = neighbor_housing_base.rename(
    columns={
        "STCOFIPS": "neighbor_fips",
        "avg_annual_housing_price": "neighbor_housing_price",
        "annual_price_growth_pct": "neighbor_price_growth_pct",
        "annual_price_volatility": "neighbor_price_volatility",
        "high_growth_flag": "neighbor_high_growth_flag",
        "high_volatility_flag": "neighbor_high_volatility_flag"
    }
)

# Merge adjacency table with neighbor housing values by neighbor county and year
county_neighbor_housing = county_adjacency.merge(
    neighbor_housing_base,
    on="neighbor_fips",
    how="left"
)

county_neighbor_housing.head()

,county_fips,county_name,neighbor_fips,neighbor_name,Year,neighbor_housing_price,neighbor_price_growth_pct,neighbor_price_volatility,neighbor_high_growth_flag,neighbor_high_volatility_flag
0,12001,Alachua,12007,Bradford,2011,102337.066909,-5.361393,995.247406,0.0,0
1,12001,Alachua,12007,Bradford,2012,101934.137211,-0.393728,1473.306233,0.0,0
2,12001,Alachua,12007,Bradford,2013,102435.128908,0.491486,1273.026499,0.0,0
3,12001,Alachua,12007,Bradford,2014,105777.819611,3.263227,955.652456,0.0,0
4,12001,Alachua,12007,Bradford,2015,106147.061966,0.349074,2059.993406,0.0,0


In [31]:
# ==================================================
# Aggregate Neighboring Housing Features
# ==================================================

neighbor_housing_features = (
    county_neighbor_housing
    .groupby(["county_fips", "Year"], as_index=False)
    .agg(
        neighbor_count=("neighbor_fips", "nunique"),
        neighbor_avg_housing_price=("neighbor_housing_price", "mean"),
        neighbor_avg_price_growth_pct=("neighbor_price_growth_pct", "mean"),
        neighbor_avg_price_volatility=("neighbor_price_volatility", "mean"),
        neighbor_high_growth_share=("neighbor_high_growth_flag", "mean"),
        neighbor_high_volatility_share=("neighbor_high_volatility_flag", "mean")
    )
)

neighbor_housing_features = neighbor_housing_features.rename(
    columns={
        "county_fips": "STCOFIPS"
    }
)

print("Neighbor housing feature shape:", neighbor_housing_features.shape)
print("Counties:", neighbor_housing_features["STCOFIPS"].nunique())
print("Years:", neighbor_housing_features["Year"].min(), "to", neighbor_housing_features["Year"].max())

neighbor_housing_features.head()

Neighbor housing feature shape: (1005, 8)
Counties: 67
Years: 2011 to 2025


,STCOFIPS,Year,neighbor_count,neighbor_avg_housing_price,neighbor_avg_price_growth_pct,neighbor_avg_price_volatility,neighbor_high_growth_share,neighbor_high_volatility_share
0,12001,2011,7,102374.849631,-6.763815,1794.893064,0.000000,0.142857
1,12001,2012,7,99827.485576,-2.455404,1117.279737,0.000000,0.000000
2,12001,2013,7,102229.301664,2.368021,1412.403491,0.142857,0.000000
3,12001,2014,7,106775.555497,4.331896,1637.672940,0.142857,0.000000
4,12001,2015,7,111968.466397,4.816762,3004.397627,0.428571,0.142857


In [32]:
# ==================================================
# Neighbor Housing Feature Missing Value Check
# ==================================================

neighbor_housing_features.isna().sum()

STCOFIPS                          0
Year                              0
neighbor_count                    0
neighbor_avg_housing_price        0
neighbor_avg_price_growth_pct     0
neighbor_avg_price_volatility     0
neighbor_high_growth_share        0
neighbor_high_volatility_share    0
dtype: int64

In [33]:
# ==================================================
# Merge Neighboring Features with County-Year Housing Data
# ==================================================

county_year_housing_spatial = county_year_housing_final.merge(
    neighbor_housing_features,
    on=["STCOFIPS", "Year"],
    how="left"
)

print("County-year housing with spatial features:", county_year_housing_spatial.shape)
print("Counties:", county_year_housing_spatial["STCOFIPS"].nunique())
print("Years:", county_year_housing_spatial["Year"].min(), "to", county_year_housing_spatial["Year"].max())
print("Duplicate county-year rows:", county_year_housing_spatial.duplicated(subset=["STCOFIPS", "Year"]).sum())

county_year_housing_spatial[
    [
        "RegionName",
        "Year",
        "avg_annual_housing_price",
        "annual_price_growth_pct",
        "neighbor_count",
        "neighbor_avg_housing_price",
        "neighbor_avg_price_growth_pct",
        "neighbor_high_growth_share"
    ]
].head()

County-year housing with spatial features: (1000, 37)
Counties: 67
Years: 2011 to 2025
Duplicate county-year rows: 0


,RegionName,Year,avg_annual_housing_price,annual_price_growth_pct,neighbor_count,neighbor_avg_housing_price,neighbor_avg_price_growth_pct,neighbor_high_growth_share
0,Alachua County,2011,144811.606954,-10.680050,7,102374.849631,-6.763815,0.000000
1,Alachua County,2012,136832.426441,-5.510042,7,99827.485576,-2.455404,0.000000
2,Alachua County,2013,140015.149500,2.326001,7,102229.301664,2.368021,0.142857
3,Alachua County,2014,147354.521716,5.241842,7,106775.555497,4.331896,0.142857
4,Alachua County,2015,153750.122064,4.340281,7,111968.466397,4.816762,0.428571


In [34]:
# ==================================================
# Spatial Housing Feature Validation
# ==================================================

spatial_feature_cols = [
    "neighbor_count",
    "neighbor_avg_housing_price",
    "neighbor_avg_price_growth_pct",
    "neighbor_avg_price_volatility",
    "neighbor_high_growth_share",
    "neighbor_high_volatility_share"
]

print("Missing values in spatial features:")
print(county_year_housing_spatial[spatial_feature_cols].isna().sum())

print("\nNeighbor count summary:")
print(county_year_housing_spatial["neighbor_count"].describe())

Missing values in spatial features:
neighbor_count                    0
neighbor_avg_housing_price        0
neighbor_avg_price_growth_pct     0
neighbor_avg_price_volatility     0
neighbor_high_growth_share        0
neighbor_high_volatility_share    0
dtype: int64

Neighbor count summary:
count    1000.000000
mean        4.780000
std         1.727595
min         1.000000
25%         4.000000
50%         5.000000
75%         6.000000
max        10.000000
Name: neighbor_count, dtype: float64


### 4.5 Save Spatial Housing Feature Layer

Save the county-year housing dataset with neighboring-county spatial features for later vulnerability modelling.

In [ ]:
# ==================================================
# 4.5 Save Spatial Housing Feature Layer
# ==================================================

county_year_housing_spatial.to_csv(
    INTERIM_DATA / "county_year_housing_spatial_features.csv",
    index=False
)

neighbor_housing_features.to_csv(
    INTERIM_DATA / "neighbor_housing_features.csv",
    index=False
)

print(
    f"Saved: "
    f"{INTERIM_DATA / 'county_year_housing_spatial_features.csv'}"
)

print(
    f"Saved: "
    f"{INTERIM_DATA / 'neighbor_housing_features.csv'}"
)

Saved: ..\data\processed\county_year_housing_spatial_features.csv
Saved: ..\data\processed\neighbor_housing_features.csv


## 5. Disaster History Data Integration

This section adds FEMA disaster declaration history to the county-year housing-spatial dataset. Disaster declarations provide a time-varying climate stress layer that captures actual federally declared disaster exposure across Florida counties.

### 5.1 Load FEMA Disaster Declarations Data

Load the FEMA Disaster Declarations dataset and inspect its structure before filtering to Florida and the project period.

In [39]:
# ==================================================
# 5.1 Load FEMA Disaster Declarations Data
# ==================================================

# FEMA Disaster Declarations file path
FEMA_DISASTER_PATH = (
    RAW_DATA
    / "fema_disaster_declarations"
    / "DisasterDeclarationsSummaries.csv"
)

# Load FEMA disaster declarations
fema_disasters = pd.read_csv(
    FEMA_DISASTER_PATH,
    dtype={
        "state": str,
        "fipsStateCode": str,
        "fipsCountyCode": str,
        "placeCode": str
    },
    low_memory=False
)

# Convert key date columns to datetime
date_cols = [
    "declarationDate",
    "incidentBeginDate",
    "incidentEndDate",
    "disasterCloseoutDate",
    "lastIAFilingDate",
    "lastRefresh"
]

for col in date_cols:
    if col in fema_disasters.columns:
        fema_disasters[col] = pd.to_datetime(fema_disasters[col], errors="coerce")

# Basic dataset overview
print("FEMA disaster declarations shape:", fema_disasters.shape)
print("Number of columns:", fema_disasters.shape[1])
print("Number of states/territories:", fema_disasters["state"].nunique())

print("\nDate range based on incident begin date:")
print("Start:", fema_disasters["incidentBeginDate"].min())
print("End:", fema_disasters["incidentBeginDate"].max())

print("\nTop incident types:")
print(fema_disasters["incidentType"].value_counts().head(10))

print("\nColumns:")
print(fema_disasters.columns.tolist())

# Preview
fema_disasters.head()

FEMA disaster declarations shape: (69936, 28)
Number of columns: 28
Number of states/territories: 59

Date range based on incident begin date:
Start: 1953-05-02 00:00:00+00:00
End: 2026-06-16 00:00:00+00:00

Top incident types:
incidentType
Severe Storm        19311
Hurricane           13726
Flood               11288
Biological           7857
Fire                 3879
Snowstorm            3707
Severe Ice Storm     2956
Tornado              1623
Winter Storm         1376
Drought              1292
Name: count, dtype: int64

Columns:
['femaDeclarationString', 'disasterNumber', 'state', 'declarationType', 'declarationDate', 'fyDeclared', 'incidentType', 'declarationTitle', 'ihProgramDeclared', 'iaProgramDeclared', 'paProgramDeclared', 'hmProgramDeclared', 'incidentBeginDate', 'incidentEndDate', 'disasterCloseoutDate', 'tribalRequest', 'fipsStateCode', 'fipsCountyCode', 'placeCode', 'designatedArea', 'declarationRequestNumber', 'lastIAFilingDate', 'incidentId', 'region', 'designatedIncident

,femaDeclarationString,disasterNumber,state,declarationType,declarationDate,fyDeclared,incidentType,declarationTitle,ihProgramDeclared,iaProgramDeclared,...,placeCode,designatedArea,declarationRequestNumber,lastIAFilingDate,incidentId,region,designatedIncidentTypes,lastRefresh,hash,id
0,EM-3610-PR,3610,PR,EM,2024-08-13 00:00:00+00:00,2024,Severe Storm,TROPICAL STORM ERNESTO,0,0,...,99001,Adjuntas (Municipio),24124,NaT,2024080901,2,"4,M,W,Z,F",2026-04-27 18:01:29.390000+00:00,ccd089a290654da031dfd82dd3a626b757e6166e,002e8dfc-2a00-49c3-915c-b4cbd2b301b7
1,FM-5529-OR,5529,OR,FM,2024-08-09 00:00:00+00:00,2024,Fire,LEE FALLS FIRE,0,0,...,99067,Washington (County),24122,NaT,2024081001,10,R,2024-08-27 18:22:14.800000+00:00,ae87cf3c6ed795015b714af7166c7c295b2b67c7,09e3f81a-5e16-4b72-b317-1c64e0cfa59c
2,FM-5528-OR,5528,OR,FM,2024-08-06 00:00:00+00:00,2024,Fire,ELK LANE FIRE,0,0,...,99031,Jefferson (County),24116,NaT,2024080701,10,R,2024-08-27 18:22:14.800000+00:00,432cf0995c47e3895cea696ede5621b810460501,59983f89-30bf-4888-b21b-62e8d57d9aac
3,FM-5527-OR,5527,OR,FM,2024-08-02 00:00:00+00:00,2024,Fire,MILE MARKER 132 FIRE,0,0,...,99017,Deschutes (County),24111,NaT,2024080301,10,R,2024-08-27 18:22:14.800000+00:00,2f21d90cb6bc64b0d4121aa3f18d852bbb4b11fa,8d13ecf0-bc2f-496b-8c9f-b2e73da832a0
4,EM-3610-PR,3610,PR,EM,2024-08-13 00:00:00+00:00,2024,Severe Storm,TROPICAL STORM ERNESTO,0,0,...,99003,Aguada (Municipio),24124,NaT,2024080901,2,"4,M,W,Z,F",2026-04-27 18:01:29.390000+00:00,0beaa37210d2cf6c22408616fd69f66339aa81db,ade80f2f-aca5-44b3-a6de-d78e2c8bb58b


### 5.2 Filter Disaster Data to Florida

Filter FEMA disaster declarations to Florida and create a county FIPS identifier so the disaster data can later be merged with the county-year housing-spatial dataset.

In [40]:
# ==================================================
# 5.2 Filter Disaster Data to Florida
# ==================================================

# Filter to Florida disaster declarations
fl_disasters = fema_disasters[fema_disasters["state"] == "FL"].copy()

# Standardize FIPS fields
fl_disasters["fipsStateCode"] = fl_disasters["fipsStateCode"].astype(str).str.zfill(2)
fl_disasters["fipsCountyCode"] = fl_disasters["fipsCountyCode"].astype(str).str.zfill(3)

# Create county FIPS identifier
fl_disasters["STCOFIPS"] = (
    fl_disasters["fipsStateCode"] + fl_disasters["fipsCountyCode"]
)

# Basic Florida disaster overview
print("Florida disaster declarations shape:", fl_disasters.shape)
print("Unique Florida counties/areas:", fl_disasters["STCOFIPS"].nunique())

print("\nFlorida disaster date range:")
print("Start:", fl_disasters["incidentBeginDate"].min())
print("End:", fl_disasters["incidentBeginDate"].max())

print("\nTop Florida incident types:")
print(fl_disasters["incidentType"].value_counts())

print("\nSample Florida disaster records:")
fl_disasters[
    [
        "disasterNumber",
        "state",
        "declarationType",
        "incidentType",
        "declarationTitle",
        "incidentBeginDate",
        "incidentEndDate",
        "designatedArea",
        "fipsStateCode",
        "fipsCountyCode",
        "STCOFIPS"
    ]
].head()

Florida disaster declarations shape: (2794, 29)
Unique Florida counties/areas: 68

Florida disaster date range:
Start: 1953-10-22 00:00:00+00:00
End: 2026-04-21 00:00:00+00:00

Top Florida incident types:
incidentType
Hurricane         1432
Severe Storm       376
Fire               280
Tropical Storm     266
Biological         150
Freezing           147
Flood               74
Tornado             38
Coastal Storm       25
Human Cause          4
Other                2
Name: count, dtype: int64

Sample Florida disaster records:


,disasterNumber,state,declarationType,incidentType,declarationTitle,incidentBeginDate,incidentEndDate,designatedArea,fipsStateCode,fipsCountyCode,STCOFIPS
130,5426,FL,FM,Fire,CHIPOLA FIRE COMPLEX,2022-03-04 00:00:00+00:00,2022-04-01 00:00:00+00:00,Bay (County),12,005,12005
131,5426,FL,FM,Fire,CHIPOLA FIRE COMPLEX,2022-03-04 00:00:00+00:00,2022-04-01 00:00:00+00:00,Calhoun (County),12,013,12013
134,5424,FL,FM,Fire,1707 ADKINS AVE FIRE,2022-03-04 00:00:00+00:00,2022-04-01 00:00:00+00:00,Bay (County),12,005,12005
299,5309,FL,FM,Fire,36TH AVENUE FIRE,2020-05-13 00:00:00+00:00,2020-06-02 00:00:00+00:00,Collier (County),12,021,12021
300,5308,FL,FM,Fire,MUSSETT BAYOU FIRE,2020-05-06 00:00:00+00:00,2020-06-02 00:00:00+00:00,Walton (County),12,131,12131


Before filtering FEMA disaster declarations, reload the Step 4 spatial housing dataset so county coverage can be validated against the final housing-spatial layer.

In [ ]:
# ==================================================
# Reload Step 4 Spatial Housing Dataset
# ==================================================

SPATIAL_HOUSING_PATH = INTERIM_DATA / "county_year_housing_spatial_features.csv"

county_year_spatial = pd.read_csv(
    SPATIAL_HOUSING_PATH,
    dtype={"STCOFIPS": str}
)

county_year_spatial["STCOFIPS"] = county_year_spatial["STCOFIPS"].str.zfill(5)
county_year_spatial["Year"] = county_year_spatial["Year"].astype(int)

print("Spatial housing dataset shape:", county_year_spatial.shape)
print("Unique counties:", county_year_spatial["STCOFIPS"].nunique())
print("Year range:", county_year_spatial["Year"].min(), "-", county_year_spatial["Year"].max())
print("Duplicate county-year rows:", county_year_spatial.duplicated(["STCOFIPS", "Year"]).sum())

county_year_spatial.head()

Spatial housing dataset shape: (1000, 37)
Unique counties: 67
Year range: 2011 - 2025
Duplicate county-year rows: 0


,STCOFIPS,RegionID,RegionName,State,Metro,StateCodeFIPS,MunicipalCodeFIPS,COUNTY,Year,avg_annual_housing_price,...,price_growth_acceleration,high_growth_flag,high_volatility_flag,complete_year_flag,neighbor_count,neighbor_avg_housing_price,neighbor_avg_price_growth_pct,neighbor_avg_price_volatility,neighbor_high_growth_share,neighbor_high_volatility_share
0,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2011,144811.606954,...,NaN,0.0,1,1,7,102374.849631,-6.763815,1794.893064,0.000000,0.142857
1,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2012,136832.426441,...,5.170008,0.0,0,1,7,99827.485576,-2.455404,1117.279737,0.000000,0.000000
2,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2013,140015.149500,...,7.836043,0.0,0,1,7,102229.301664,2.368021,1412.403491,0.142857,0.000000
3,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2014,147354.521716,...,2.915841,0.0,0,1,7,106775.555497,4.331896,1637.672940,0.142857,0.000000
4,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2015,153750.122064,...,-0.901560,0.0,0,1,7,111968.466397,4.816762,3004.397627,0.428571,0.142857


### 5.3 Filter to Project Period

Filter Florida disaster declarations to the project period, 2011–2025, and remove non-county or statewide records so the disaster data matches the county-year housing-spatial dataset.

In [43]:
# ==================================================
# 5.3 Filter Disaster Data to Project Period
# ==================================================

# Create disaster year using incident begin date
fl_disasters["Year"] = fl_disasters["incidentBeginDate"].dt.year

# Filter to project period
fl_disasters_project = fl_disasters[
    (fl_disasters["Year"] >= 2011) &
    (fl_disasters["Year"] <= 2025)
].copy()

# Remove statewide or non-county records
# Florida county FIPS codes should be 12001 to 12133 and should match the 67 housing counties
housing_county_fips = set(county_year_spatial["STCOFIPS"].astype(str).unique())

fl_disasters_project = fl_disasters_project[
    fl_disasters_project["STCOFIPS"].isin(housing_county_fips)
].copy()

# Basic validation
print("Florida disaster declarations during project period:", fl_disasters_project.shape)
print("Unique counties in disaster data:", fl_disasters_project["STCOFIPS"].nunique())
print("Project years in disaster data:", fl_disasters_project["Year"].min(), "-", fl_disasters_project["Year"].max())

print("\nIncident types during project period:")
print(fl_disasters_project["incidentType"].value_counts())

print("\nDeclarations by year:")
print(fl_disasters_project["Year"].value_counts().sort_index())

print("\nCounty coverage check:")
print("Counties in housing-spatial dataset:", county_year_spatial["STCOFIPS"].nunique())
print("Counties with at least one disaster declaration:", fl_disasters_project["STCOFIPS"].nunique())
print("Counties with no disaster declaration during project period:", 
      len(housing_county_fips - set(fl_disasters_project["STCOFIPS"].unique())))

# Preview filtered records
fl_disasters_project[
    [
        "disasterNumber",
        "Year",
        "STCOFIPS",
        "designatedArea",
        "declarationType",
        "incidentType",
        "declarationTitle",
        "incidentBeginDate",
        "incidentEndDate"
    ]
].head()

Florida disaster declarations during project period: (1329, 30)
Unique counties in disaster data: 67
Project years in disaster data: 2011 - 2024

Incident types during project period:
incidentType
Hurricane         829
Tropical Storm    257
Biological        134
Severe Storm       97
Fire               10
Other               1
Flood               1
Name: count, dtype: int64

Declarations by year:
Year
2011      1
2012     46
2013      4
2014      9
2016     72
2017    139
2018     53
2019     81
2020    196
2021     61
2022    243
2023     99
2024    325
Name: count, dtype: int64

County coverage check:
Counties in housing-spatial dataset: 67
Counties with at least one disaster declaration: 67
Counties with no disaster declaration during project period: 0


,disasterNumber,Year,STCOFIPS,designatedArea,declarationType,incidentType,declarationTitle,incidentBeginDate,incidentEndDate
130,5426,2022,12005,Bay (County),FM,Fire,CHIPOLA FIRE COMPLEX,2022-03-04 00:00:00+00:00,2022-04-01 00:00:00+00:00
131,5426,2022,12013,Calhoun (County),FM,Fire,CHIPOLA FIRE COMPLEX,2022-03-04 00:00:00+00:00,2022-04-01 00:00:00+00:00
134,5424,2022,12005,Bay (County),FM,Fire,1707 ADKINS AVE FIRE,2022-03-04 00:00:00+00:00,2022-04-01 00:00:00+00:00
299,5309,2020,12021,Collier (County),FM,Fire,36TH AVENUE FIRE,2020-05-13 00:00:00+00:00,2020-06-02 00:00:00+00:00
300,5308,2020,12131,Walton (County),FM,Fire,MUSSETT BAYOU FIRE,2020-05-06 00:00:00+00:00,2020-06-02 00:00:00+00:00


### 5.4 Create County-Year Disaster Indicators

Aggregate FEMA disaster declarations into county-year indicators. These variables capture annual disaster exposure, climate-related disaster exposure, and recent disaster burden for each Florida county.

In [44]:
# ==================================================
# 5.4 Create County-Year Disaster Indicators
# ==================================================

# Create a working copy
disaster_work = fl_disasters_project.copy()

# Standardize incident type text
disaster_work["incidentType"] = disaster_work["incidentType"].astype(str).str.strip()

# Define climate-related disaster categories
climate_incident_types = [
    "Hurricane",
    "Tropical Storm",
    "Severe Storm",
    "Flood",
    "Coastal Storm",
    "Fire",
    "Tornado",
    "Freezing"
]

# Create incident-type flags
disaster_work["climate_disaster_flag"] = disaster_work["incidentType"].isin(climate_incident_types).astype(int)
disaster_work["hurricane_disaster_flag"] = (disaster_work["incidentType"] == "Hurricane").astype(int)
disaster_work["tropical_storm_disaster_flag"] = (disaster_work["incidentType"] == "Tropical Storm").astype(int)
disaster_work["severe_storm_disaster_flag"] = (disaster_work["incidentType"] == "Severe Storm").astype(int)
disaster_work["flood_disaster_flag"] = (disaster_work["incidentType"] == "Flood").astype(int)
disaster_work["fire_disaster_flag"] = (disaster_work["incidentType"] == "Fire").astype(int)

# Aggregate to county-year level
county_year_disasters = (
    disaster_work
    .groupby(["STCOFIPS", "Year"])
    .agg(
        disaster_count=("disasterNumber", "count"),
        unique_disaster_events=("disasterNumber", "nunique"),
        climate_disaster_count=("climate_disaster_flag", "sum"),
        hurricane_disaster_count=("hurricane_disaster_flag", "sum"),
        tropical_storm_disaster_count=("tropical_storm_disaster_flag", "sum"),
        severe_storm_disaster_count=("severe_storm_disaster_flag", "sum"),
        flood_disaster_count=("flood_disaster_flag", "sum"),
        fire_disaster_count=("fire_disaster_flag", "sum")
    )
    .reset_index()
)

# Create any-disaster flags
county_year_disasters["any_disaster_flag"] = (
    county_year_disasters["disaster_count"] > 0
).astype(int)

county_year_disasters["any_climate_disaster_flag"] = (
    county_year_disasters["climate_disaster_count"] > 0
).astype(int)

# Create a complete county-year grid from the housing-spatial dataset
county_year_grid = county_year_spatial[["STCOFIPS", "Year"]].drop_duplicates().copy()

# Merge disaster indicators onto the full county-year grid
county_year_disasters = county_year_grid.merge(
    county_year_disasters,
    on=["STCOFIPS", "Year"],
    how="left"
)

# Fill county-years with no FEMA declarations as 0
disaster_cols = [
    "disaster_count",
    "unique_disaster_events",
    "climate_disaster_count",
    "hurricane_disaster_count",
    "tropical_storm_disaster_count",
    "severe_storm_disaster_count",
    "flood_disaster_count",
    "fire_disaster_count",
    "any_disaster_flag",
    "any_climate_disaster_flag"
]

county_year_disasters[disaster_cols] = county_year_disasters[disaster_cols].fillna(0)

# Convert disaster columns to integers
county_year_disasters[disaster_cols] = county_year_disasters[disaster_cols].astype(int)

# Sort for rolling and cumulative calculations
county_year_disasters = county_year_disasters.sort_values(["STCOFIPS", "Year"]).reset_index(drop=True)

# Create cumulative disaster burden
county_year_disasters["cumulative_disaster_count"] = (
    county_year_disasters
    .groupby("STCOFIPS")["disaster_count"]
    .cumsum()
)

county_year_disasters["cumulative_climate_disaster_count"] = (
    county_year_disasters
    .groupby("STCOFIPS")["climate_disaster_count"]
    .cumsum()
)

# Create recent 3-year disaster burden
county_year_disasters["recent_3yr_disaster_count"] = (
    county_year_disasters
    .groupby("STCOFIPS")["disaster_count"]
    .transform(lambda x: x.rolling(window=3, min_periods=1).sum())
)

county_year_disasters["recent_3yr_climate_disaster_count"] = (
    county_year_disasters
    .groupby("STCOFIPS")["climate_disaster_count"]
    .transform(lambda x: x.rolling(window=3, min_periods=1).sum())
)

# Convert rolling columns to integers
rolling_cols = [
    "cumulative_disaster_count",
    "cumulative_climate_disaster_count",
    "recent_3yr_disaster_count",
    "recent_3yr_climate_disaster_count"
]

county_year_disasters[rolling_cols] = county_year_disasters[rolling_cols].astype(int)

# Validation summary
print("County-year disaster indicators shape:", county_year_disasters.shape)
print("Unique counties:", county_year_disasters["STCOFIPS"].nunique())
print("Year range:", county_year_disasters["Year"].min(), "-", county_year_disasters["Year"].max())
print("Duplicate county-year rows:", county_year_disasters.duplicated(["STCOFIPS", "Year"]).sum())

print("\nDisaster indicator summary:")
print(county_year_disasters[
    [
        "disaster_count",
        "climate_disaster_count",
        "hurricane_disaster_count",
        "tropical_storm_disaster_count",
        "severe_storm_disaster_count",
        "flood_disaster_count",
        "recent_3yr_disaster_count",
        "cumulative_disaster_count"
    ]
].describe())

county_year_disasters.head()

County-year disaster indicators shape: (1000, 16)
Unique counties: 67
Year range: 2011 - 2025
Duplicate county-year rows: 0

Disaster indicator summary:
       disaster_count  climate_disaster_count  hurricane_disaster_count  \
count     1000.000000             1000.000000               1000.000000   
mean         1.328000                1.193000                  0.828000   
std          1.551394                1.492642                  1.010662   
min          0.000000                0.000000                  0.000000   
25%          0.000000                0.000000                  0.000000   
50%          1.000000                1.000000                  0.000000   
75%          2.000000                2.000000                  2.000000   
max          7.000000                7.000000                  3.000000   

       tropical_storm_disaster_count  severe_storm_disaster_count  \
count                    1000.000000                  1000.000000   
mean                        0.257

,STCOFIPS,Year,disaster_count,unique_disaster_events,climate_disaster_count,hurricane_disaster_count,tropical_storm_disaster_count,severe_storm_disaster_count,flood_disaster_count,fire_disaster_count,any_disaster_flag,any_climate_disaster_flag,cumulative_disaster_count,cumulative_climate_disaster_count,recent_3yr_disaster_count,recent_3yr_climate_disaster_count
0,12001,2011,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,12001,2012,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,12001,2013,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,12001,2014,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,12001,2015,0,0,0,0,0,0,0,0,0,0,0,0,0,0


### 5.5 Merge Disaster Indicators with Housing-Spatial Dataset

Merge the county-year FEMA disaster indicators with the spatial housing dataset to create a disaster-enhanced climate-housing vulnerability dataset.

In [45]:
# ==================================================
# 5.5 Merge Disaster Indicators with Housing-Spatial Dataset
# ==================================================

# Merge disaster indicators with spatial housing dataset
county_year_disaster_enhanced = county_year_spatial.merge(
    county_year_disasters,
    on=["STCOFIPS", "Year"],
    how="left"
)

# Disaster columns expected after merge
disaster_feature_cols = [
    "disaster_count",
    "unique_disaster_events",
    "climate_disaster_count",
    "hurricane_disaster_count",
    "tropical_storm_disaster_count",
    "severe_storm_disaster_count",
    "flood_disaster_count",
    "fire_disaster_count",
    "any_disaster_flag",
    "any_climate_disaster_flag",
    "cumulative_disaster_count",
    "cumulative_climate_disaster_count",
    "recent_3yr_disaster_count",
    "recent_3yr_climate_disaster_count"
]

# Fill any missing disaster values with 0
county_year_disaster_enhanced[disaster_feature_cols] = (
    county_year_disaster_enhanced[disaster_feature_cols]
    .fillna(0)
    .astype(int)
)

# Basic merge validation
print("Spatial housing dataset shape:", county_year_spatial.shape)
print("Disaster indicators shape:", county_year_disasters.shape)
print("Disaster-enhanced dataset shape:", county_year_disaster_enhanced.shape)

print("\nUnique counties:", county_year_disaster_enhanced["STCOFIPS"].nunique())
print("Year range:", county_year_disaster_enhanced["Year"].min(), "-", county_year_disaster_enhanced["Year"].max())
print("Duplicate county-year rows:", county_year_disaster_enhanced.duplicated(["STCOFIPS", "Year"]).sum())

print("\nMissing values in disaster features:")
print(county_year_disaster_enhanced[disaster_feature_cols].isna().sum())

# Preview selected columns
county_year_disaster_enhanced[
    [
        "STCOFIPS",
        "RegionName",
        "Year",
        "avg_annual_housing_price",
        "annual_price_growth_pct",
        "neighbor_avg_housing_price",
        "CFLD_RISKS",
        "HRCN_RISKS",
        "SOVI_SCORE",
        "RESL_SCORE",
        "disaster_count",
        "climate_disaster_count",
        "hurricane_disaster_count",
        "recent_3yr_disaster_count",
        "cumulative_disaster_count"
    ]
].head()

Spatial housing dataset shape: (1000, 37)
Disaster indicators shape: (1000, 16)
Disaster-enhanced dataset shape: (1000, 51)

Unique counties: 67
Year range: 2011 - 2025
Duplicate county-year rows: 0

Missing values in disaster features:
disaster_count                       0
unique_disaster_events               0
climate_disaster_count               0
hurricane_disaster_count             0
tropical_storm_disaster_count        0
severe_storm_disaster_count          0
flood_disaster_count                 0
fire_disaster_count                  0
any_disaster_flag                    0
any_climate_disaster_flag            0
cumulative_disaster_count            0
cumulative_climate_disaster_count    0
recent_3yr_disaster_count            0
recent_3yr_climate_disaster_count    0
dtype: int64


,STCOFIPS,RegionName,Year,avg_annual_housing_price,annual_price_growth_pct,neighbor_avg_housing_price,CFLD_RISKS,HRCN_RISKS,SOVI_SCORE,RESL_SCORE,disaster_count,climate_disaster_count,hurricane_disaster_count,recent_3yr_disaster_count,cumulative_disaster_count
0,12001,Alachua County,2011,144811.606954,-10.680050,102374.849631,0.0,96.704214,34.764631,80.979644,0,0,0,0,0
1,12001,Alachua County,2012,136832.426441,-5.510042,99827.485576,0.0,96.704214,34.764631,80.979644,0,0,0,0,0
2,12001,Alachua County,2013,140015.149500,2.326001,102229.301664,0.0,96.704214,34.764631,80.979644,0,0,0,0,0
3,12001,Alachua County,2014,147354.521716,5.241842,106775.555497,0.0,96.704214,34.764631,80.979644,0,0,0,0,0
4,12001,Alachua County,2015,153750.122064,4.340281,111968.466397,0.0,96.704214,34.764631,80.979644,0,0,0,0,0


### 5.6 Validate Disaster-Enhanced Dataset

Validate the disaster-enhanced dataset to confirm that the merge preserved the county-year structure, maintained full county coverage, and added complete disaster-history features.

In [46]:
# ==================================================
# 5.6 Validate Disaster-Enhanced Dataset
# ==================================================

# Core structure checks
expected_rows = county_year_spatial.shape[0]
actual_rows = county_year_disaster_enhanced.shape[0]

expected_counties = county_year_spatial["STCOFIPS"].nunique()
actual_counties = county_year_disaster_enhanced["STCOFIPS"].nunique()

duplicate_county_years = county_year_disaster_enhanced.duplicated(["STCOFIPS", "Year"]).sum()

# Missing value checks
disaster_missing = county_year_disaster_enhanced[disaster_feature_cols].isna().sum()
total_missing_disaster_values = disaster_missing.sum()

# County-year coverage checks
missing_counties_after_merge = (
    set(county_year_spatial["STCOFIPS"].unique()) -
    set(county_year_disaster_enhanced["STCOFIPS"].unique())
)

missing_years_after_merge = (
    set(county_year_spatial["Year"].unique()) -
    set(county_year_disaster_enhanced["Year"].unique())
)

# Disaster exposure summaries
county_disaster_summary = (
    county_year_disaster_enhanced
    .groupby(["STCOFIPS", "RegionName"])
    .agg(
        total_disaster_count=("disaster_count", "sum"),
        total_climate_disaster_count=("climate_disaster_count", "sum"),
        total_hurricane_disaster_count=("hurricane_disaster_count", "sum"),
        max_recent_3yr_disaster_count=("recent_3yr_disaster_count", "max"),
        max_cumulative_disaster_count=("cumulative_disaster_count", "max")
    )
    .reset_index()
    .sort_values("total_disaster_count", ascending=False)
)

# Print validation results
print("Disaster-enhanced dataset validation")
print("-----------------------------------")
print("Expected rows:", expected_rows)
print("Actual rows:", actual_rows)
print("Expected counties:", expected_counties)
print("Actual counties:", actual_counties)
print("Year range:", county_year_disaster_enhanced["Year"].min(), "-", county_year_disaster_enhanced["Year"].max())
print("Duplicate county-year rows:", duplicate_county_years)

print("\nMissing disaster values:", total_missing_disaster_values)
print("Missing counties after merge:", len(missing_counties_after_merge))
print("Missing years after merge:", len(missing_years_after_merge))

print("\nDisaster feature totals:")
print(county_year_disaster_enhanced[disaster_feature_cols].sum().sort_values(ascending=False))

print("\nTop 10 counties by total disaster declarations:")
display(county_disaster_summary.head(10))

print("\nRows by year:")
print(county_year_disaster_enhanced["Year"].value_counts().sort_index())

Disaster-enhanced dataset validation
-----------------------------------
Expected rows: 1000
Actual rows: 1000
Expected counties: 67
Actual counties: 67
Year range: 2011 - 2025
Duplicate county-year rows: 0

Missing disaster values: 0
Missing counties after merge: 0
Missing years after merge: 0

Disaster feature totals:
cumulative_disaster_count            7167
cumulative_climate_disaster_count    6358
recent_3yr_disaster_count            3659
recent_3yr_climate_disaster_count    3254
disaster_count                       1328
unique_disaster_events               1328
climate_disaster_count               1193
hurricane_disaster_count              828
any_disaster_flag                     576
any_climate_disaster_flag             556
tropical_storm_disaster_count         257
severe_storm_disaster_count            97
fire_disaster_count                    10
flood_disaster_count                    1
dtype: int64

Top 10 counties by total disaster declarations:


,STCOFIPS,RegionName,total_disaster_count,total_climate_disaster_count,total_hurricane_disaster_count,max_recent_3yr_disaster_count,max_cumulative_disaster_count
8,12017,Citrus County,24,22,15,12,24
61,12123,Taylor County,24,22,14,13,24
2,12005,Bay County,23,21,13,10,23
36,12075,Levy County,23,21,14,12,23
39,12081,Manatee County,23,21,14,12,23
17,12037,Franklin County,23,21,15,9,23
10,12021,Collier County,23,21,11,12,23
25,12053,Hernando County,23,21,14,12,23
31,12065,Jefferson County,23,21,13,11,23
19,12041,Gilchrist County,23,21,14,12,23



Rows by year:
Year
2011    66
2012    66
2013    66
2014    66
2015    66
2016    67
2017    67
2018    67
2019    67
2020    67
2021    67
2022    67
2023    67
2024    67
2025    67
Name: count, dtype: int64


### 5.7 Save Disaster-Enhanced Dataset

Save the disaster-enhanced county-year dataset and the county-year disaster indicator table for later vulnerability scoring and modelling.

In [49]:
# ==================================================
# 5.7 Save Disaster-Enhanced Dataset
# ==================================================

# Output file paths
DISASTER_ENHANCED_PATH = INTERIM_DATA / "county_year_housing_spatial_disaster_features.csv"
DISASTER_INDICATORS_PATH = INTERIM_DATA / "county_year_disaster_indicators.csv"

# Save interim disaster-enhanced dataset
county_year_disaster_enhanced.to_csv(DISASTER_ENHANCED_PATH, index=False)

# Save standalone county-year disaster indicator table
county_year_disasters.to_csv(DISASTER_INDICATORS_PATH, index=False)

print("Saved interim disaster-enhanced dataset to:")
print(DISASTER_ENHANCED_PATH)

print("\nSaved county-year disaster indicators to:")
print(DISASTER_INDICATORS_PATH)

print("\nInterim disaster-enhanced dataset shape:", county_year_disaster_enhanced.shape)
print("County-year disaster indicators shape:", county_year_disasters.shape)

Saved interim disaster-enhanced dataset to:
..\data\interim\county_year_features\county_year_housing_spatial_disaster_features.csv

Saved county-year disaster indicators to:
..\data\interim\county_year_features\county_year_disaster_indicators.csv

Interim disaster-enhanced dataset shape: (1000, 51)
County-year disaster indicators shape: (1000, 16)


## 6. Socioeconomic and Housing Affordability Data Integration

This section adds ACS socioeconomic and housing affordability indicators to the county-year vulnerability dataset. These variables help capture whether counties may have less capacity to absorb climate-related housing stress.

### 6.1 Identify ACS Variables for Vulnerability Framework

Identify the ACS 5-year variables needed to represent socioeconomic vulnerability, housing tenure, and affordability pressure at the county-year level.

In [62]:
# ==================================================
# 6.1 Identify ACS Variables for Vulnerability Framework
# ==================================================

import requests
from getpass import getpass

# ACS 5-year estimates will be pulled by county for Florida.
# The project period is 2011–2025, but ACS 5-year data is currently available up to 2024.
# We will pull available ACS years first, then handle 2025 later if needed.

ACS_START_YEAR = 2011
ACS_END_YEAR = 2024

ACS_YEARS = list(range(ACS_START_YEAR, ACS_END_YEAR + 1))

# Census API base structure
ACS_BASE_URL = "https://api.census.gov/data/{year}/acs/acs5"

# Enter Census API key securely
# This avoids showing the key directly in the notebook output.
CENSUS_API_KEY = getpass("Enter your Census API key: ")

# ACS variables selected for socioeconomic and affordability indicators
acs_variables = {
    # Population
    "B01003_001E": "acs_total_population",
    
    # Income
    "B19013_001E": "median_household_income",
    
    # Poverty
    "B17001_001E": "poverty_universe",
    "B17001_002E": "poverty_count",
    
    # Employment
    "B23025_003E": "labor_force",
    "B23025_005E": "unemployed_count",
    
    # Housing tenure
    "B25003_001E": "occupied_housing_units",
    "B25003_002E": "owner_occupied_units",
    "B25003_003E": "renter_occupied_units",
    
    # Housing affordability
    "B25064_001E": "median_gross_rent",
    "B25077_001E": "acs_median_home_value"
}

# ACS output folder
ACS_OUTPUT_DIR = INTERIM_DATA / "acs"
ACS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("ACS years selected:", ACS_YEARS[0], "-", ACS_YEARS[-1])
print("Number of ACS years:", len(ACS_YEARS))
print("Number of ACS variables:", len(acs_variables))

print("\nSelected ACS variables:")
for code, name in acs_variables.items():
    print(f"{code}: {name}")

print("\nACS output folder:")
print(ACS_OUTPUT_DIR)
print("Folder exists:", ACS_OUTPUT_DIR.exists())



ACS years selected: 2011 - 2024
Number of ACS years: 14
Number of ACS variables: 11

Selected ACS variables:
B01003_001E: acs_total_population
B19013_001E: median_household_income
B17001_001E: poverty_universe
B17001_002E: poverty_count
B23025_003E: labor_force
B23025_005E: unemployed_count
B25003_001E: occupied_housing_units
B25003_002E: owner_occupied_units
B25003_003E: renter_occupied_units
B25064_001E: median_gross_rent
B25077_001E: acs_median_home_value

ACS output folder:
..\data\interim\county_year_features\acs
Folder exists: True


In [63]:
# ==================================================
# Test Census API Key
# ==================================================

test_year = 2023

test_params = {
    "get": "NAME,B01003_001E,B19013_001E",
    "for": "county:*",
    "in": "state:12",
    "key": CENSUS_API_KEY
}

test_url = ACS_BASE_URL.format(year=test_year)

response = requests.get(test_url, params=test_params, timeout=30)

print("\nACS API key test")
print("----------------")
print("Status code:", response.status_code)

# Hide key in printed URL
safe_url = response.url.replace(CENSUS_API_KEY, "HIDDEN_KEY")
print("Requested URL:")
print(safe_url)

print("\nResponse preview:")
print(response.text[:500])

if response.status_code == 200 and response.text.strip().startswith("["):
    print("\nAPI test successful. Ready for 6.2.")
else:
    print("\nAPI test failed. Check the response preview before moving to 6.2.")


ACS API key test
----------------
Status code: 200
Requested URL:
https://api.census.gov/data/2023/acs/acs5?get=NAME%2CB01003_001E%2CB19013_001E&for=county%3A%2A&in=state%3A12&key=HIDDEN_KEY

Response preview:
[["NAME","B01003_001E","B19013_001E","state","county"],
["Alachua County, Florida","281751","59659","12","001"],
["Baker County, Florida","28186","70833","12","003"],
["Bay County, Florida","181368","70188","12","005"],
["Bradford County, Florida","27888","59740","12","007"],
["Brevard County, Florida","620533","75817","12","009"],
["Broward County, Florida","1946127","74534","12","011"],
["Calhoun County, Florida","13593","46901","12","013"],
["Charlotte County, Florida","195083","66154","12","

API test successful. Ready for 6.2.


### 6.2 Load ACS County-Level Data from Census API

Pull selected ACS 5-year county-level variables for Florida from the Census API for each available ACS year.

In [64]:
# ==================================================
# 6.2 Load ACS County-Level Data from Census API
# ==================================================

import time

acs_yearly_data = []
failed_acs_years = []

# Variables to request from the Census API
acs_variable_codes = list(acs_variables.keys())

for year in ACS_YEARS:
    print(f"Pulling ACS {year} data...")
    
    # Build Census API request
    params = {
        "get": "NAME," + ",".join(acs_variable_codes),
        "for": "county:*",
        "in": "state:12",
        "key": CENSUS_API_KEY
    }
    
    url = ACS_BASE_URL.format(year=year)
    
    try:
        response = requests.get(url, params=params, timeout=30)
        
        # Check that response is valid JSON-style Census output
        if response.status_code == 200 and response.text.strip().startswith("["):
            data = response.json()
            
            # First row is header, remaining rows are data
            acs_year_df = pd.DataFrame(data[1:], columns=data[0])
            
            # Add ACS year
            acs_year_df["Year"] = year
            
            # Create county FIPS identifier
            acs_year_df["STCOFIPS"] = (
                acs_year_df["state"].astype(str).str.zfill(2) +
                acs_year_df["county"].astype(str).str.zfill(3)
            )
            
            # Rename ACS variable columns
            acs_year_df = acs_year_df.rename(columns=acs_variables)
            
            acs_yearly_data.append(acs_year_df)
            
            print(f"  Success: {acs_year_df.shape[0]} county rows")
        
        else:
            failed_acs_years.append(year)
            print(f"  Failed: status code {response.status_code}")
            print("  Response preview:")
            print(response.text[:300])
    
    except Exception as e:
        failed_acs_years.append(year)
        print(f"  Failed: {e}")
    
    # Small pause to avoid sending requests too quickly
    time.sleep(0.25)

# Combine all successful years into one dataframe
if len(acs_yearly_data) > 0:
    acs_county_year_raw = pd.concat(acs_yearly_data, ignore_index=True)
else:
    raise ValueError("No ACS data was successfully downloaded. Check API key or variable codes.")

# Convert selected ACS columns to numeric
acs_numeric_cols = list(acs_variables.values())

for col in acs_numeric_cols:
    acs_county_year_raw[col] = pd.to_numeric(
        acs_county_year_raw[col],
        errors="coerce"
    )

# Keep useful columns only
acs_keep_cols = [
    "STCOFIPS",
    "Year",
    "NAME"
] + acs_numeric_cols

acs_county_year_raw = acs_county_year_raw[acs_keep_cols].copy()

# Save raw ACS pull
ACS_RAW_PATH = ACS_OUTPUT_DIR / "acs_county_year_raw_2011_2024.csv"
acs_county_year_raw.to_csv(ACS_RAW_PATH, index=False)

# Basic output checks
print("\nACS county-year raw dataset shape:", acs_county_year_raw.shape)
print("Unique counties:", acs_county_year_raw["STCOFIPS"].nunique())
print("Year range:", acs_county_year_raw["Year"].min(), "-", acs_county_year_raw["Year"].max())
print("Duplicate county-year rows:", acs_county_year_raw.duplicated(["STCOFIPS", "Year"]).sum())

print("\nFailed ACS years:", failed_acs_years)

print("\nMissing values by ACS variable:")
print(acs_county_year_raw[acs_numeric_cols].isna().sum())

print("\nSaved raw ACS data to:")
print(ACS_RAW_PATH)

acs_county_year_raw.head()

Pulling ACS 2011 data...
  Success: 67 county rows
Pulling ACS 2012 data...
  Success: 67 county rows
Pulling ACS 2013 data...
  Success: 67 county rows
Pulling ACS 2014 data...
  Success: 67 county rows
Pulling ACS 2015 data...
  Success: 67 county rows
Pulling ACS 2016 data...
  Success: 67 county rows
Pulling ACS 2017 data...
  Success: 67 county rows
Pulling ACS 2018 data...
  Success: 67 county rows
Pulling ACS 2019 data...
  Success: 67 county rows
Pulling ACS 2020 data...
  Success: 67 county rows
Pulling ACS 2021 data...
  Success: 67 county rows
Pulling ACS 2022 data...
  Success: 67 county rows
Pulling ACS 2023 data...
  Success: 67 county rows
Pulling ACS 2024 data...
  Success: 67 county rows

ACS county-year raw dataset shape: (938, 14)
Unique counties: 67
Year range: 2011 - 2024
Duplicate county-year rows: 0

Failed ACS years: []

Missing values by ACS variable:
acs_total_population       0
median_household_income    0
poverty_universe           0
poverty_count           

,STCOFIPS,Year,NAME,acs_total_population,median_household_income,poverty_universe,poverty_count,labor_force,unemployed_count,occupied_housing_units,owner_occupied_units,renter_occupied_units,median_gross_rent,acs_median_home_value
0,12085,2011,"Martin County, Florida",145480,53612,142450,15397,66987,7173,59316,46746,12570,992,238200
1,12115,2011,"Sarasota County, Florida",378404,49212,372765,40886,169529,18202,169256,128934,40322,1013,213400
2,12017,2011,"Citrus County, Florida",141157,38189,138733,21922,53093,7956,59915,49936,9979,770,134800
3,12113,2011,"Santa Rosa County, Florida",150073,55913,146588,15868,71691,7155,55956,42695,13261,956,173400
4,12011,2011,"Broward County, Florida",1742012,51782,1725510,223485,944554,99062,665037,453419,211618,1162,225300


### 6.3 Clean and Prepare ACS Variables

This step cleans the raw ACS county-year dataset and creates socioeconomic context indicators needed for the vulnerability framework.

The goal is to convert raw ACS counts into interpretable county-year indicators such as poverty rate, unemployment rate, renter share, and owner-occupied share.

In [65]:
# ==================================================
# 6.3 Clean and Prepare ACS Variables
# ==================================================

# Load raw ACS county-year data
acs_raw_path = ACS_OUTPUT_DIR / "acs_county_year_raw_2011_2024.csv"

acs_clean = pd.read_csv(acs_raw_path)

print("Raw ACS dataset shape:", acs_clean.shape)
print("Unique counties:", acs_clean["STCOFIPS"].nunique())
print("Year range:", acs_clean["Year"].min(), "to", acs_clean["Year"].max())

# --------------------------------------------------
# Ensure county FIPS is stored as a 5-digit string
# --------------------------------------------------

acs_clean["STCOFIPS"] = acs_clean["STCOFIPS"].astype(str).str.zfill(5)

# --------------------------------------------------
# Convert ACS numeric columns
# --------------------------------------------------

acs_numeric_cols = [
    "acs_total_population",
    "median_household_income",
    "poverty_universe",
    "poverty_count",
    "labor_force",
    "unemployed_count",
    "occupied_housing_units",
    "owner_occupied_units",
    "renter_occupied_units",
    "median_gross_rent",
    "acs_median_home_value"
]

for col in acs_numeric_cols:
    acs_clean[col] = pd.to_numeric(acs_clean[col], errors="coerce")

# --------------------------------------------------
# Create ACS-derived socioeconomic indicators
# --------------------------------------------------

acs_clean["poverty_rate"] = (
    acs_clean["poverty_count"] / acs_clean["poverty_universe"]
) * 100

acs_clean["unemployment_rate"] = (
    acs_clean["unemployed_count"] / acs_clean["labor_force"]
) * 100

acs_clean["owner_occupied_share"] = (
    acs_clean["owner_occupied_units"] / acs_clean["occupied_housing_units"]
) * 100

acs_clean["renter_occupied_share"] = (
    acs_clean["renter_occupied_units"] / acs_clean["occupied_housing_units"]
) * 100

# --------------------------------------------------
# Keep ACS-derived indicators separate for validation
# --------------------------------------------------

acs_derived_cols = [
    "poverty_rate",
    "unemployment_rate",
    "owner_occupied_share",
    "renter_occupied_share"
]

# --------------------------------------------------
# Preview cleaned ACS data
# --------------------------------------------------

print("\nClean ACS dataset shape:", acs_clean.shape)

print("\nMissing values in ACS-derived indicators:")
print(acs_clean[acs_derived_cols].isna().sum())

print("\nSummary of ACS-derived indicators:")
display(acs_clean[acs_derived_cols].describe().round(2))

display(acs_clean.head())

Raw ACS dataset shape: (938, 14)
Unique counties: 67
Year range: 2011 to 2024

Clean ACS dataset shape: (938, 18)

Missing values in ACS-derived indicators:
poverty_rate             0
unemployment_rate        0
owner_occupied_share     0
renter_occupied_share    0
dtype: int64

Summary of ACS-derived indicators:


,poverty_rate,unemployment_rate,owner_occupied_share,renter_occupied_share
count,938.00,938.00,938.00,938.00
mean,16.50,8.18,72.64,27.36
std,5.23,3.40,7.65,7.65
min,5.59,2.42,51.16,9.60
25%,12.38,5.35,69.15,22.26
50%,15.49,7.49,74.18,25.82
75%,19.96,10.62,77.74,30.85
max,31.87,22.75,90.40,48.84


,STCOFIPS,Year,NAME,acs_total_population,median_household_income,poverty_universe,poverty_count,labor_force,unemployed_count,occupied_housing_units,owner_occupied_units,renter_occupied_units,median_gross_rent,acs_median_home_value,poverty_rate,unemployment_rate,owner_occupied_share,renter_occupied_share
0,12085,2011,"Martin County, Florida",145480,53612,142450,15397,66987,7173,59316,46746,12570,992,238200,10.808705,10.708048,78.808416,21.191584
1,12115,2011,"Sarasota County, Florida",378404,49212,372765,40886,169529,18202,169256,128934,40322,1013,213400,10.968304,10.736806,76.176915,23.823085
2,12017,2011,"Citrus County, Florida",141157,38189,138733,21922,53093,7956,59915,49936,9979,770,134800,15.801576,14.985026,83.344738,16.655262
3,12113,2011,"Santa Rosa County, Florida",150073,55913,146588,15868,71691,7155,55956,42695,13261,956,173400,10.824897,9.980332,76.301022,23.698978
4,12011,2011,"Broward County, Florida",1742012,51782,1725510,223485,944554,99062,665037,453419,211618,1162,225300,12.951823,10.487701,68.179515,31.820485


#### 6.3 Output Interpretation

The ACS cleaning step successfully created four socioeconomic and housing-tenure indicators for all county-year observations from 2011 to 2024.

No missing values were found in the derived ACS indicators, confirming that the raw ACS variables were complete enough to support the vulnerability framework.

The average poverty rate across Florida county-years was approximately 16.5%, while the average unemployment rate was approximately 8.2%. Housing tenure patterns show that owner-occupied households were more common than renter-occupied households, with an average owner-occupied share of approximately 72.6%.

These variables will help represent socioeconomic vulnerability and housing stability in the climate-housing vulnerability framework.

### 6.4 Validate and Save Clean ACS Features

This step validates the cleaned ACS county-year dataset before merging it with the main housing-spatial-disaster dataset.

The purpose is to confirm that the ACS layer has complete county-year coverage, no duplicate county-year rows, no missing derived indicators, and reasonable percentage ranges.

In [66]:
# ==================================================
# 6.4 Validate and Save Clean ACS Features
# ==================================================

# --------------------------------------------------
# Define expected ACS coverage
# --------------------------------------------------

expected_acs_years = list(range(ACS_START_YEAR, ACS_END_YEAR + 1))
expected_acs_rows = 67 * len(expected_acs_years)

print("Expected ACS rows:", expected_acs_rows)
print("Actual ACS rows:", len(acs_clean))

print("\nExpected ACS year range:", min(expected_acs_years), "to", max(expected_acs_years))
print("Actual ACS year range:", acs_clean["Year"].min(), "to", acs_clean["Year"].max())

print("\nUnique counties:", acs_clean["STCOFIPS"].nunique())
print("Unique years:", acs_clean["Year"].nunique())

# --------------------------------------------------
# Check duplicate county-year rows
# --------------------------------------------------

acs_duplicate_rows = acs_clean.duplicated(subset=["STCOFIPS", "Year"]).sum()

print("\nDuplicate county-year rows:", acs_duplicate_rows)

# --------------------------------------------------
# Check missing values in key ACS variables
# --------------------------------------------------

acs_key_cols = [
    "acs_total_population",
    "median_household_income",
    "poverty_universe",
    "poverty_count",
    "labor_force",
    "unemployed_count",
    "occupied_housing_units",
    "owner_occupied_units",
    "renter_occupied_units",
    "median_gross_rent",
    "acs_median_home_value",
    "poverty_rate",
    "unemployment_rate",
    "owner_occupied_share",
    "renter_occupied_share"
]

print("\nMissing values in key ACS variables:")
print(acs_clean[acs_key_cols].isna().sum())

# --------------------------------------------------
# Check rate ranges
# --------------------------------------------------

acs_rate_cols = [
    "poverty_rate",
    "unemployment_rate",
    "owner_occupied_share",
    "renter_occupied_share"
]

print("\nMinimum values for ACS rate variables:")
print(acs_clean[acs_rate_cols].min().round(2))

print("\nMaximum values for ACS rate variables:")
print(acs_clean[acs_rate_cols].max().round(2))

# --------------------------------------------------
# Check owner/renter share consistency
# --------------------------------------------------

acs_clean["housing_tenure_share_total"] = (
    acs_clean["owner_occupied_share"] + acs_clean["renter_occupied_share"]
)

print("\nOwner + renter share total summary:")
display(acs_clean["housing_tenure_share_total"].describe().round(4))

# --------------------------------------------------
# Final validation summary
# --------------------------------------------------

acs_validation_summary = pd.DataFrame({
    "Check": [
        "Expected rows",
        "Actual rows",
        "Unique counties",
        "Unique years",
        "Duplicate county-year rows",
        "Missing key ACS values",
        "Minimum housing tenure share total",
        "Maximum housing tenure share total"
    ],
    "Value": [
        expected_acs_rows,
        len(acs_clean),
        acs_clean["STCOFIPS"].nunique(),
        acs_clean["Year"].nunique(),
        acs_duplicate_rows,
        acs_clean[acs_key_cols].isna().sum().sum(),
        round(acs_clean["housing_tenure_share_total"].min(), 4),
        round(acs_clean["housing_tenure_share_total"].max(), 4)
    ]
})

display(acs_validation_summary)

# --------------------------------------------------
# Save cleaned ACS feature dataset
# --------------------------------------------------

acs_clean_path = ACS_OUTPUT_DIR / "acs_county_year_clean_2011_2024.csv"

acs_clean.to_csv(acs_clean_path, index=False)

print("\nSaved cleaned ACS feature dataset to:")
print(acs_clean_path)

Expected ACS rows: 938
Actual ACS rows: 938

Expected ACS year range: 2011 to 2024
Actual ACS year range: 2011 to 2024

Unique counties: 67
Unique years: 14

Duplicate county-year rows: 0

Missing values in key ACS variables:
acs_total_population       0
median_household_income    0
poverty_universe           0
poverty_count              0
labor_force                0
unemployed_count           0
occupied_housing_units     0
owner_occupied_units       0
renter_occupied_units      0
median_gross_rent          0
acs_median_home_value      0
poverty_rate               0
unemployment_rate          0
owner_occupied_share       0
renter_occupied_share      0
dtype: int64

Minimum values for ACS rate variables:
poverty_rate              5.59
unemployment_rate         2.42
owner_occupied_share     51.16
renter_occupied_share     9.60
dtype: float64

Maximum values for ACS rate variables:
poverty_rate             31.87
unemployment_rate        22.75
owner_occupied_share     90.40
renter_occupie

count    938.0
mean     100.0
std        0.0
min      100.0
25%      100.0
50%      100.0
75%      100.0
max      100.0
Name: housing_tenure_share_total, dtype: float64

,Check,Value
0,Expected rows,938.0
1,Actual rows,938.0
2,Unique counties,67.0
3,Unique years,14.0
4,Duplicate county-year rows,0.0
5,Missing key ACS values,0.0
6,Minimum housing tenure share total,100.0
7,Maximum housing tenure share total,100.0



Saved cleaned ACS feature dataset to:
..\data\interim\county_year_features\acs\acs_county_year_clean_2011_2024.csv


#### 6.4 Output Interpretation

The cleaned ACS feature dataset was successfully validated before merging with the main county-year dataset.

The dataset contains the expected 938 county-year observations, covering 67 Florida counties from 2011 to 2024. No duplicate county-year rows were found, and no missing values were present in the key ACS variables or derived socioeconomic indicators.

The derived rate variables fall within reasonable ranges. The owner-occupied and renter-occupied shares sum to 100% for all observations, confirming that the housing tenure indicators were calculated correctly.

This confirms that the ACS layer is ready to be merged with the housing-spatial-disaster dataset in the next step.

### 6.5 Merge ACS Features with Disaster-Enhanced Dataset

This step merges the cleaned ACS socioeconomic indicators with the housing-spatial-disaster county-year dataset.

The ACS layer adds socioeconomic vulnerability and housing affordability context to the project. After merging, the price-to-income ratio is calculated using Zillow annual housing prices and ACS median household income.

Because ACS 2025 5-year estimates are not available yet, 2025 county-year rows are expected to have missing ACS values after the merge.

In [69]:
# ==================================================
# 6.5 Merge ACS Features with Disaster-Enhanced Dataset
# ==================================================

# --------------------------------------------------
# Load disaster-enhanced county-year dataset
# --------------------------------------------------

disaster_enhanced_path = INTERIM_DATA / "county_year_housing_spatial_disaster_features.csv"

county_year_disaster = pd.read_csv(disaster_enhanced_path)

print("Disaster-enhanced dataset shape:", county_year_disaster.shape)
print("Unique counties:", county_year_disaster["STCOFIPS"].nunique())
print("Year range:", county_year_disaster["Year"].min(), "to", county_year_disaster["Year"].max())

# --------------------------------------------------
# Ensure merge keys are aligned
# --------------------------------------------------

county_year_disaster["STCOFIPS"] = county_year_disaster["STCOFIPS"].astype(str).str.zfill(5)
acs_clean["STCOFIPS"] = acs_clean["STCOFIPS"].astype(str).str.zfill(5)

county_year_disaster["Year"] = county_year_disaster["Year"].astype(int)
acs_clean["Year"] = acs_clean["Year"].astype(int)

# --------------------------------------------------
# Select ACS columns needed for the vulnerability dataset
# --------------------------------------------------

acs_feature_cols = [
    "STCOFIPS",
    "Year",
    "acs_total_population",
    "median_household_income",
    "poverty_rate",
    "unemployment_rate",
    "owner_occupied_share",
    "renter_occupied_share",
    "median_gross_rent",
    "acs_median_home_value"
]

acs_features = acs_clean[acs_feature_cols].copy()

print("\nACS feature dataset shape:", acs_features.shape)
print("ACS year range:", acs_features["Year"].min(), "to", acs_features["Year"].max())

# --------------------------------------------------
# Merge ACS features with disaster-enhanced dataset
# --------------------------------------------------

county_year_acs = county_year_disaster.merge(
    acs_features,
    on=["STCOFIPS", "Year"],
    how="left"
)

print("\nMerged county-year ACS dataset shape:", county_year_acs.shape)
print("Unique counties:", county_year_acs["STCOFIPS"].nunique())
print("Year range:", county_year_acs["Year"].min(), "to", county_year_acs["Year"].max())

# --------------------------------------------------
# Create affordability indicator
# --------------------------------------------------

county_year_acs["price_to_income_ratio"] = (
    county_year_acs["avg_annual_housing_price"] / county_year_acs["median_household_income"]
)

# --------------------------------------------------
# Check ACS merge coverage
# --------------------------------------------------

acs_merged_cols = [
    "acs_total_population",
    "median_household_income",
    "poverty_rate",
    "unemployment_rate",
    "owner_occupied_share",
    "renter_occupied_share",
    "median_gross_rent",
    "acs_median_home_value",
    "price_to_income_ratio"
]

print("\nMissing values after ACS merge:")
print(county_year_acs[acs_merged_cols].isna().sum())

print("\nMissing ACS values by year:")
display(
    county_year_acs
    .assign(missing_acs=county_year_acs["median_household_income"].isna())
    .groupby("Year")["missing_acs"]
    .sum()
    .reset_index()
)

# --------------------------------------------------
# Validate duplicate county-year rows
# --------------------------------------------------

duplicate_county_year_rows = county_year_acs.duplicated(
    subset=["STCOFIPS", "Year"]
).sum()

print("\nDuplicate county-year rows after ACS merge:", duplicate_county_year_rows)

# --------------------------------------------------
# Preview affordability indicator
# --------------------------------------------------

print("\nPrice-to-income ratio summary:")
display(county_year_acs["price_to_income_ratio"].describe().round(2))

# Check available county/name columns
print("\nAvailable county/name-related columns:")
print([col for col in county_year_acs.columns if "county" in col.lower() or "name" in col.lower()])

# Preview key ACS-affordability variables
preview_cols = [
    "STCOFIPS",
    "Year",
    "avg_annual_housing_price",
    "median_household_income",
    "price_to_income_ratio",
    "poverty_rate",
    "unemployment_rate",
    "owner_occupied_share",
    "renter_occupied_share"
]

display(county_year_acs[preview_cols].head())

Disaster-enhanced dataset shape: (1000, 51)
Unique counties: 67
Year range: 2011 to 2025

ACS feature dataset shape: (938, 10)
ACS year range: 2011 to 2024

Merged county-year ACS dataset shape: (1000, 59)
Unique counties: 67
Year range: 2011 to 2025

Missing values after ACS merge:
acs_total_population       67
median_household_income    67
poverty_rate               67
unemployment_rate          67
owner_occupied_share       67
renter_occupied_share      67
median_gross_rent          67
acs_median_home_value      67
price_to_income_ratio      67
dtype: int64

Missing ACS values by year:


,Year,missing_acs
0,2011,0
1,2012,0
2,2013,0
3,2014,0
4,2015,0
5,2016,0
6,2017,0
7,2018,0
8,2019,0
9,2020,0



Duplicate county-year rows after ACS merge: 0

Price-to-income ratio summary:


count    933.00
mean       3.86
std        1.26
min        1.86
25%        2.92
50%        3.66
75%        4.50
max       11.82
Name: price_to_income_ratio, dtype: float64


Available county/name-related columns:
['RegionName', 'COUNTY']


,STCOFIPS,Year,avg_annual_housing_price,median_household_income,price_to_income_ratio,poverty_rate,unemployment_rate,owner_occupied_share,renter_occupied_share
0,12001,2011,144811.606954,41373.0,3.500148,23.557117,7.332445,54.467819,45.532181
1,12001,2012,136832.426441,42818.0,3.195675,23.784030,7.865880,55.316747,44.683253
2,12001,2013,140015.149500,42149.0,3.321909,24.900498,8.640740,54.144498,45.855502
3,12001,2014,147354.521716,42045.0,3.504686,25.362068,8.535418,53.540260,46.459740
4,12001,2015,153750.122064,43073.0,3.569524,24.342127,7.858113,53.202072,46.797928


#### 6.5 Output Interpretation

The cleaned ACS socioeconomic features were successfully merged with the housing-spatial-disaster county-year dataset.

The merged dataset retained the full project structure of 1,000 county-year observations, 67 Florida counties, and the 2011–2025 study period. No duplicate county-year rows were created during the merge.

ACS variables were successfully matched for all available years from 2011 to 2024. As expected, the 2025 rows contain missing ACS values because ACS 2025 5-year estimates are not yet available.

The price-to-income ratio was created by dividing average annual housing price by median household income. This variable captures housing affordability pressure and will later contribute to the vulnerability framework.

The average price-to-income ratio across available county-years was approximately 3.86, with values ranging from 1.86 to 11.82. Higher values indicate counties where housing prices are high relative to local household income.

### 6.6 Estimate 2025 ACS Values Using Recent County Trends

ACS 2025 5-year estimates are not available yet, which created missing ACS values for all 2025 county-year rows.

To preserve the full 2011–2025 project period, this step estimates 2025 ACS values using a county-specific linear trend based on the most recent five available ACS years from 2020 to 2024.

This approach is used instead of a complex machine-learning imputation model because only one year is missing, the missingness is expected, and ACS variables are treated as slow-moving socioeconomic context indicators.

In [70]:
# ==================================================
# 6.6 Estimate 2025 ACS Values Using Recent County Trends
# ==================================================

# --------------------------------------------------
# Create working copy
# --------------------------------------------------

county_year_acs_projected = county_year_acs.copy()

# --------------------------------------------------
# Define ACS variables to project for 2025
# --------------------------------------------------

acs_projection_cols = [
    "acs_total_population",
    "median_household_income",
    "poverty_rate",
    "unemployment_rate",
    "owner_occupied_share",
    "renter_occupied_share",
    "median_gross_rent",
    "acs_median_home_value"
]

projection_base_years = [2020, 2021, 2022, 2023, 2024]
projection_year = 2025

# --------------------------------------------------
# Add flag for estimated ACS values
# --------------------------------------------------

county_year_acs_projected["acs_estimated_2025_flag"] = 0

# --------------------------------------------------
# Check missing ACS values before projection
# --------------------------------------------------

print("Missing ACS values before 2025 projection:")
print(county_year_acs_projected[acs_projection_cols].isna().sum())

# --------------------------------------------------
# Estimate 2025 ACS values using county-specific trends
# --------------------------------------------------

projection_records = []

for county_fips in county_year_acs_projected["STCOFIPS"].unique():
    
    county_mask = county_year_acs_projected["STCOFIPS"] == county_fips
    county_data = county_year_acs_projected.loc[county_mask].copy()
    
    for col in acs_projection_cols:
        
        # Use recent available ACS values from 2020 to 2024
        recent_data = county_data[
            county_data["Year"].isin(projection_base_years)
        ][["Year", col]].dropna()
        
        # Only project if the 2025 value is missing
        missing_2025_mask = (
            (county_year_acs_projected["STCOFIPS"] == county_fips) &
            (county_year_acs_projected["Year"] == projection_year) &
            (county_year_acs_projected[col].isna())
        )
        
        if missing_2025_mask.sum() == 1:
            
            # Fit linear trend if enough recent data exists
            if len(recent_data) >= 2:
                trend_coefficients = np.polyfit(
                    recent_data["Year"],
                    recent_data[col],
                    deg=1
                )
                
                projected_value = np.polyval(
                    trend_coefficients,
                    projection_year
                )
            
            # Fallback: use 2024 value if trend cannot be estimated
            else:
                projected_value = county_data.loc[
                    county_data["Year"] == 2024,
                    col
                ].iloc[0]
            
            # Keep percentage variables within valid range
            if col in [
                "poverty_rate",
                "unemployment_rate",
                "owner_occupied_share",
                "renter_occupied_share"
            ]:
                projected_value = np.clip(projected_value, 0, 100)
            
            # Keep count/value variables non-negative
            else:
                projected_value = max(projected_value, 0)
            
            # Fill projected value
            county_year_acs_projected.loc[missing_2025_mask, col] = projected_value
            
            # Store projection record
            projection_records.append({
                "STCOFIPS": county_fips,
                "Variable": col,
                "Projected_Year": projection_year,
                "Projected_Value": projected_value
            })

# --------------------------------------------------
# Mark 2025 ACS rows as estimated
# --------------------------------------------------

county_year_acs_projected.loc[
    county_year_acs_projected["Year"] == projection_year,
    "acs_estimated_2025_flag"
] = 1

# --------------------------------------------------
# Recalculate affordability indicator after projection
# --------------------------------------------------

county_year_acs_projected["price_to_income_ratio"] = (
    county_year_acs_projected["avg_annual_housing_price"] /
    county_year_acs_projected["median_household_income"]
)

# --------------------------------------------------
# Create projection summary table
# --------------------------------------------------

acs_projection_summary = pd.DataFrame(projection_records)

print("\nProjection records created:", len(acs_projection_summary))

print("\nProjected ACS variables:")
print(acs_projection_summary["Variable"].value_counts())

# --------------------------------------------------
# Validate missing values after projection
# --------------------------------------------------

acs_final_cols = acs_projection_cols + ["price_to_income_ratio"]

print("\nMissing ACS values after 2025 projection:")
print(county_year_acs_projected[acs_final_cols].isna().sum())

print("\nACS estimated 2025 flag counts:")
print(county_year_acs_projected["acs_estimated_2025_flag"].value_counts())

# --------------------------------------------------
# Validate duplicate county-year rows
# --------------------------------------------------

duplicate_county_year_rows = county_year_acs_projected.duplicated(
    subset=["STCOFIPS", "Year"]
).sum()

print("\nDuplicate county-year rows after ACS projection:", duplicate_county_year_rows)

# --------------------------------------------------
# Preview projected 2025 ACS values
# --------------------------------------------------

print("\nPreview of projected 2025 ACS values:")

display(
    county_year_acs_projected[
        county_year_acs_projected["Year"] == 2025
    ][
        [
            "STCOFIPS",
            "Year",
            "avg_annual_housing_price",
            "median_household_income",
            "price_to_income_ratio",
            "poverty_rate",
            "unemployment_rate",
            "owner_occupied_share",
            "renter_occupied_share",
            "acs_estimated_2025_flag"
        ]
    ].head()
)

# --------------------------------------------------
# Save ACS-enhanced county-year dataset
# --------------------------------------------------

acs_enhanced_path = INTERIM_DATA / "county_year_housing_spatial_disaster_acs_features.csv"

county_year_acs_projected.to_csv(acs_enhanced_path, index=False)

print("\nSaved ACS-enhanced county-year dataset to:")
print(acs_enhanced_path)

Missing ACS values before 2025 projection:
acs_total_population       67
median_household_income    67
poverty_rate               67
unemployment_rate          67
owner_occupied_share       67
renter_occupied_share      67
median_gross_rent          67
acs_median_home_value      67
dtype: int64

Projection records created: 536

Projected ACS variables:
Variable
acs_total_population       67
median_household_income    67
poverty_rate               67
unemployment_rate          67
owner_occupied_share       67
renter_occupied_share      67
median_gross_rent          67
acs_median_home_value      67
Name: count, dtype: int64

Missing ACS values after 2025 projection:
acs_total_population       0
median_household_income    0
poverty_rate               0
unemployment_rate          0
owner_occupied_share       0
renter_occupied_share      0
median_gross_rent          0
acs_median_home_value      0
price_to_income_ratio      0
dtype: int64

ACS estimated 2025 flag counts:
acs_estimated_2025_f

,STCOFIPS,Year,avg_annual_housing_price,median_household_income,price_to_income_ratio,poverty_rate,unemployment_rate,owner_occupied_share,renter_occupied_share,acs_estimated_2025_flag
14,12001,2025,299662.702434,65534.9,4.572567,20.505281,4.201802,53.864632,46.135368,1
29,12003,2025,301416.508099,81554.1,3.695909,13.239940,4.196839,87.354278,12.645722,1
44,12005,2025,342660.873614,78479.7,4.366236,11.289835,3.479108,67.646335,32.353665,1
59,12007,2025,237914.832325,68949.5,3.450566,16.605275,5.613864,74.551136,25.448864,1
74,12009,2025,341438.431783,84844.1,4.024304,9.621525,4.734424,77.274480,22.725520,1



Saved ACS-enhanced county-year dataset to:
..\data\interim\county_year_features\county_year_housing_spatial_disaster_acs_features.csv


#### 6.6 Output Interpretation

The missing 2025 ACS values were successfully estimated using county-specific linear trends based on the most recent five available ACS years from 2020 to 2024.

Before projection, each ACS variable had 67 missing values, corresponding to the 67 Florida counties in 2025. After projection, all ACS variables and the price-to-income ratio had zero missing values.

A total of 536 projected ACS values were created, representing 67 counties across 8 ACS variables. The `acs_estimated_2025_flag` was added to identify rows where ACS values were estimated rather than directly observed.

The final ACS-enhanced dataset retains the full 2011–2025 project period, contains no duplicate county-year rows, and is ready for the next feature layer.

## 7. NFIP / Insurance Data Feasibility

The next layer considered for the vulnerability framework is flood-insurance and insurance-related financial risk.

This step explores whether National Flood Insurance Program (NFIP) data can be integrated into the county-year dataset. Insurance data is important because climate risk may affect housing markets through insurance costs, claim payments, coverage gaps, and repeated flood-loss exposure before these risks are fully reflected in housing prices.

The goal of this step is first to evaluate feasibility. If a suitable NFIP dataset can be matched to Florida counties and the project period, it will be aggregated into county-year features and merged with the ACS-enhanced dataset.

### 7.1 Identify Candidate NFIP / Insurance Datasets

Several FEMA/OpenFEMA datasets were identified as possible sources for the insurance and flood-loss layer.

The candidate datasets are:

1. **NFIP Residential Penetration Rates**  
   This dataset provides NFIP take-up rates, or the estimated percentage of residential structures covered by an NFIP policy. This is useful for measuring flood-insurance coverage and possible insurance protection gaps.

2. **FIMA NFIP Redacted Claims**  
   This dataset provides details on NFIP claims transactions. It may be useful for measuring insured flood-loss history, claim counts, and claim payments by county-year.

3. **FIMA NFIP Redacted Policies**  
   This dataset provides details on NFIP policy transactions. It may be useful for measuring policy exposure, insurance coverage, premiums, and policy activity, but it may require careful aggregation because it is transaction-based.

4. **NFIP Multiple Loss Properties**  
   This dataset provides information on structures with multiple NFIP claims. It may be useful for identifying repeated flood-loss exposure, although it may be less directly suited to annual county-year modelling.

The initial feasibility priority is:

1. NFIP Residential Penetration Rates  
2. NFIP Redacted Claims  
3. NFIP Redacted Policies  
4. NFIP Multiple Loss Properties  

This order prioritizes datasets that are more likely to provide interpretable county-level insurance or flood-loss indicators for the vulnerability framework.

In [72]:
# ==================================================
# 7.1 Identify Candidate NFIP / Insurance Datasets
# ==================================================

nfip_candidate_datasets = pd.DataFrame({
    "Priority": [1, 2, 3, 4],
    "Dataset": [
        "NFIP Residential Penetration Rates",
        "FIMA NFIP Redacted Claims",
        "FIMA NFIP Redacted Policies",
        "NFIP Multiple Loss Properties"
    ],
    "Main Purpose": [
        "Flood insurance coverage / take-up rate",
        "Flood insurance claim history and payments",
        "Flood insurance policy exposure and activity",
        "Repeated flood-loss property exposure"
    ],
    "Potential Variables": [
        "nfip_penetration_rate, insured_residential_structures, total_residential_structures",
        "nfip_claim_count, total_claim_payment, avg_claim_payment, cumulative_claims",
        "policy_count, premium_amount, coverage_amount, policy_growth",
        "multiple_loss_property_count, repetitive_loss_indicator"
    ],
    "Expected Feasibility": [
        "High if county-level fields and time fields are available",
        "Medium because claims data may be large but useful",
        "Medium/Hard because policy transactions require careful aggregation",
        "Medium if county-level geography is available"
    ],
    "Use Decision": [
        "Explore first",
        "Explore second",
        "Explore third",
        "Explore if needed"
    ]
})

display(nfip_candidate_datasets)

,Priority,Dataset,Main Purpose,Potential Variables,Expected Feasibility,Use Decision
0,1,NFIP Residential Penetration Rates,Flood insurance coverage / take-up rate,"nfip_penetration_rate, insured_residential_str...",High if county-level fields and time fields ar...,Explore first
1,2,FIMA NFIP Redacted Claims,Flood insurance claim history and payments,"nfip_claim_count, total_claim_payment, avg_cla...",Medium because claims data may be large but us...,Explore second
2,3,FIMA NFIP Redacted Policies,Flood insurance policy exposure and activity,"policy_count, premium_amount, coverage_amount,...",Medium/Hard because policy transactions requir...,Explore third
3,4,NFIP Multiple Loss Properties,Repeated flood-loss property exposure,"multiple_loss_property_count, repetitive_loss_...",Medium if county-level geography is available,Explore if needed


### 7.2 Review OpenFEMA Metadata and Available Fields

This step reviews the available OpenFEMA NFIP dataset endpoints before downloading or integrating any data.

The purpose is to check whether each candidate dataset contains the fields needed for county-year matching, such as geography, state, county, date, year, policy, claim, payment, or penetration-rate variables.

Only small sample requests are used at this stage so that large NFIP datasets are not downloaded unnecessarily.

### 7.2A Test NFIP Residential Penetration Rates Endpoint

This step tests the NFIP Residential Penetration Rates endpoint first because it is the highest-priority insurance dataset for the feasibility review.

This dataset is useful because it may provide flood-insurance take-up or penetration-rate information, which can help measure how much of a county’s residential housing stock is covered by NFIP policies.

Only a small sample is requested at this stage to inspect the response structure and available fields before deciding whether the dataset can be integrated.

In [74]:
# ==================================================
# 7.2A Test NFIP Residential Penetration Rates Endpoint
# ==================================================

import requests
import json

# --------------------------------------------------
# Test first-priority dataset: NFIP Residential Penetration Rates
# --------------------------------------------------

penetration_endpoint = "https://www.fema.gov/api/open/v1/NfipResidentialPenetrationRates"

params = {
    "$top": 5
}

response = requests.get(penetration_endpoint, params=params, timeout=60)

print("Request URL:")
print(response.url)

print("\nStatus code:")
print(response.status_code)

print("\nResponse content type:")
print(response.headers.get("Content-Type"))

print("\nFirst 500 characters of response:")
print(response.text[:500])

Request URL:
https://www.fema.gov/api/open/v1/NfipResidentialPenetrationRates?%24top=5

Status code:
200

Response content type:
application/json; charset=utf-8

First 500 characters of response:
{"metadata": {"skip":0,"select":null,"rundate":"2026-06-22T04:32:49.855Z","top":5,"filter":"","format":"json","metadata":true,"orderby":"","entityname":"NfipResidentialPenetrationRates","version":"v1","url":"/api/open/v1/NfipResidentialPenetrationRates?%24top=5","count":0}, "NfipResidentialPenetrationRates": [{"state":"Alabama","county":"Autauga","resPenetrationRateSfha":0.2261,"resPenetrationRate":0.0091,"resContractsInForceSfha":142,"resContractsInForce":189,"totalResStructuresSfha":628,"total


### 7.2B Inspect NFIP Residential Penetration Rate Fields

After confirming that the NFIP Residential Penetration Rates endpoint is accessible, this step inspects the JSON structure and available fields.

The purpose is to determine whether the dataset contains the geographic and time fields needed for integration into the county-year vulnerability dataset.

In [75]:
# ==================================================
# 7.2B Inspect NFIP Residential Penetration Rate Fields
# ==================================================

# --------------------------------------------------
# Convert response to JSON
# --------------------------------------------------

penetration_json = response.json()

# --------------------------------------------------
# Inspect top-level JSON keys
# --------------------------------------------------

print("Top-level JSON keys:")
print(penetration_json.keys())

# --------------------------------------------------
# Inspect metadata
# --------------------------------------------------

print("\nMetadata:")
display(pd.DataFrame([penetration_json["metadata"]]))

# --------------------------------------------------
# Extract records
# --------------------------------------------------

penetration_records = penetration_json["NfipResidentialPenetrationRates"]

print("\nNumber of records returned:")
print(len(penetration_records))

# --------------------------------------------------
# Convert sample records to dataframe
# --------------------------------------------------

penetration_sample = pd.DataFrame(penetration_records)

print("\nSample shape:")
print(penetration_sample.shape)

print("\nAvailable columns:")
print(penetration_sample.columns.tolist())

display(penetration_sample.head())

Top-level JSON keys:
dict_keys(['metadata', 'NfipResidentialPenetrationRates'])

Metadata:


,skip,select,rundate,top,filter,format,metadata,orderby,entityname,version,url,count
0,0,None,2026-06-22T04:32:49.855Z,5,,json,True,,NfipResidentialPenetrationRates,v1,/api/open/v1/NfipResidentialPenetrationRates?%...,0



Number of records returned:
5

Sample shape:
(5, 11)

Available columns:
['state', 'county', 'resPenetrationRateSfha', 'resPenetrationRate', 'resContractsInForceSfha', 'resContractsInForce', 'totalResStructuresSfha', 'totalResStructures', 'fipsCode', 'asOfDate', 'id']


,state,county,resPenetrationRateSfha,resPenetrationRate,resContractsInForceSfha,resContractsInForce,totalResStructuresSfha,totalResStructures,fipsCode,asOfDate,id
0,Alabama,Autauga,0.2261,0.0091,142,189,628,20789,01001,2026-04-29T00:00:00.000Z,b5fbcbe4-dd9d-4f9c-a344-a2284c4198af
1,Alabama,Baldwin,0.6494,0.0947,6781,9226,10442,97390,01003,2026-04-29T00:00:00.000Z,24c92319-ed4d-4cb4-b530-c88b2c2ba653
2,Alabama,Barbour,0.1282,0.0030,20,31,156,10455,01005,2026-04-29T00:00:00.000Z,f8906eb8-e73f-45ec-aedc-0b3db09ac6b3
3,Alabama,Bibb,0.0488,0.0021,12,16,246,7693,01007,2026-04-29T00:00:00.000Z,8d4aa492-3f9d-41ce-a99b-709d362cb604
4,Alabama,Blount,0.0885,0.0011,17,24,192,22586,01009,2026-04-29T00:00:00.000Z,3213f254-b7e5-453b-a6cd-2a6758d4e0a3


### 7.2C Explore Florida Coverage in NFIP Residential Penetration Rates

After confirming that the NFIP Residential Penetration Rates endpoint is accessible, this step filters the dataset to Florida and checks whether it can support county-year integration.

The main questions are:

- Does the dataset include all 67 Florida counties?
- Does it include county FIPS codes that match the project dataset?
- Does it contain multiple `asOfDate` values or only one current snapshot?
- Can it be used as a time-varying county-year feature, or only as a static county-level insurance coverage layer?

In [76]:
# ==================================================
# 7.2C Explore Florida Coverage in NFIP Residential Penetration Rates
# ==================================================

# --------------------------------------------------
# Request Florida records from the NFIP Residential Penetration Rates endpoint
# --------------------------------------------------

penetration_endpoint = "https://www.fema.gov/api/open/v1/NfipResidentialPenetrationRates"

params = {
    "$filter": "state eq 'Florida'",
    "$top": 5000
}

response_fl = requests.get(
    penetration_endpoint,
    params=params,
    timeout=60
)

print("Request URL:")
print(response_fl.url)

print("\nStatus code:")
print(response_fl.status_code)

# --------------------------------------------------
# Convert response to JSON and extract records
# --------------------------------------------------

penetration_fl_json = response_fl.json()

print("\nTop-level JSON keys:")
print(penetration_fl_json.keys())

penetration_fl_records = penetration_fl_json["NfipResidentialPenetrationRates"]

penetration_fl = pd.DataFrame(penetration_fl_records)

print("\nFlorida NFIP penetration dataset shape:")
print(penetration_fl.shape)

print("\nAvailable columns:")
print(penetration_fl.columns.tolist())

# --------------------------------------------------
# Preview Florida records
# --------------------------------------------------

display(penetration_fl.head())

# --------------------------------------------------
# Check Florida county and date coverage
# --------------------------------------------------

print("\nUnique states:")
print(penetration_fl["state"].unique())

print("\nUnique Florida counties:")
print(penetration_fl["county"].nunique())

print("\nUnique FIPS codes:")
print(penetration_fl["fipsCode"].nunique())

print("\nUnique asOfDate values:")
print(penetration_fl["asOfDate"].nunique())

print("\nAs-of dates:")
print(sorted(penetration_fl["asOfDate"].dropna().unique()))

# --------------------------------------------------
# Check whether Florida county FIPS matches the project dataset
# --------------------------------------------------

penetration_fl["STCOFIPS"] = penetration_fl["fipsCode"].astype(str).str.zfill(5)

project_counties = set(county_year_acs_projected["STCOFIPS"].unique())
penetration_counties = set(penetration_fl["STCOFIPS"].unique())

missing_from_penetration = sorted(project_counties - penetration_counties)
extra_in_penetration = sorted(penetration_counties - project_counties)

print("\nProject counties:", len(project_counties))
print("NFIP penetration counties:", len(penetration_counties))
print("Matched counties:", len(project_counties & penetration_counties))

print("\nMissing project counties from NFIP penetration dataset:")
print(missing_from_penetration)

print("\nExtra NFIP counties not in project dataset:")
print(extra_in_penetration)

Request URL:
https://www.fema.gov/api/open/v1/NfipResidentialPenetrationRates?%24filter=state+eq+%27Florida%27&%24top=5000

Status code:
200

Top-level JSON keys:
dict_keys(['metadata', 'NfipResidentialPenetrationRates'])

Florida NFIP penetration dataset shape:
(68, 11)

Available columns:
['state', 'county', 'resPenetrationRateSfha', 'resPenetrationRate', 'resContractsInForceSfha', 'resContractsInForce', 'totalResStructuresSfha', 'totalResStructures', 'fipsCode', 'asOfDate', 'id']


,state,county,resPenetrationRateSfha,resPenetrationRate,resContractsInForceSfha,resContractsInForce,totalResStructuresSfha,totalResStructures,fipsCode,asOfDate,id
0,Florida,Alachua,0.1931,0.0264,782,2230,4050.0,84597.0,12001,2026-04-29T00:00:00.000Z,bbafaeca-f936-493f-8fe5-17689cb9f2eb
1,Florida,Baker,0.2157,0.0153,99,152,459.0,9954.0,12003,2026-04-29T00:00:00.000Z,73949d04-f6f7-4a41-b95d-1a46a573b674
2,Florida,Bay,0.6904,0.1828,6835,13698,9900.0,74934.0,12005,2026-04-29T00:00:00.000Z,36669b64-4c56-44e3-ada8-d4ec22dd7989
3,Florida,Bradford,0.2688,0.0402,354,437,1317.0,10879.0,12007,2026-04-29T00:00:00.000Z,f1df510c-2f11-45d5-87ba-79513a05d1ab
4,Florida,Brevard,0.4795,0.1215,8184,30277,17068.0,249193.0,12009,2026-04-29T00:00:00.000Z,1cb7369b-1fec-4285-bacb-6d28aeb619c9



Unique states:
<StringArray>
['Florida']
Length: 1, dtype: str

Unique Florida counties:
68

Unique FIPS codes:
67

Unique asOfDate values:
1

As-of dates:
['2026-04-29T00:00:00.000Z']

Project counties: 67
NFIP penetration counties: 68
Matched counties: 67

Missing project counties from NFIP penetration dataset:
[]

Extra NFIP counties not in project dataset:
[nan]


#### 7.2C Output Interpretation

The NFIP Residential Penetration Rates endpoint was successfully queried for Florida records.

The filtered Florida dataset returned 68 records, with 67 unique valid FIPS codes. All 67 Florida counties in the project dataset were matched using the `fipsCode` field, and no project counties were missing from the NFIP penetration dataset. One extra record appeared with a missing FIPS value, which can be removed if this dataset is used later.

The dataset includes useful county-level insurance coverage variables, including residential penetration rate, residential penetration rate within SFHA areas, contracts in force, residential structures, and total residential structures.

However, the dataset contains only one unique `asOfDate`, dated 2026-04-29. This indicates that the dataset is a current snapshot rather than a historical county-year dataset. Therefore, it cannot directly provide time-varying insurance variables for 2011–2025.

The dataset remains useful as a possible static county-level flood-insurance coverage layer, but it should not be treated as an annual historical insurance dataset.

### 7.2D Test NFIP Redacted Claims Endpoint

After reviewing the NFIP Residential Penetration Rates dataset, the next candidate dataset is the FIMA NFIP Redacted Claims dataset.

This dataset is important because it may contain historical flood-insurance claim records, including dates of loss, claim payments, state, county, and FIPS information. If these fields are available, the claims data may support county-year flood-loss indicators such as claim counts, total claim payments, average claim payments, recent claim burden, and cumulative claim burden.

Only a small sample is requested at this stage to inspect the response structure and available fields before deciding whether full integration is feasible.

In [77]:
# ==================================================
# 7.2D Test NFIP Redacted Claims Endpoint
# ==================================================

# --------------------------------------------------
# Test second-priority dataset: NFIP Redacted Claims
# --------------------------------------------------

claims_endpoint = "https://www.fema.gov/api/open/v2/FimaNfipClaims"

params = {
    "$top": 5
}

claims_response = requests.get(
    claims_endpoint,
    params=params,
    timeout=60
)

print("Request URL:")
print(claims_response.url)

print("\nStatus code:")
print(claims_response.status_code)

print("\nResponse content type:")
print(claims_response.headers.get("Content-Type"))

print("\nFirst 500 characters of response:")
print(claims_response.text[:500])

Request URL:
https://www.fema.gov/api/open/v2/FimaNfipClaims?%24top=5

Status code:
200

Response content type:
application/json; charset=utf-8

First 500 characters of response:
{"metadata": {"skip":0,"select":null,"rundate":"2026-06-22T04:52:32.510Z","top":5,"filter":"","format":"json","metadata":true,"orderby":"","entityname":"FimaNfipClaims","version":"v2","url":"/api/open/v2/FimaNfipClaims?%24top=5","count":0}, "FimaNfipClaims": [{"agricultureStructureIndicator":false,"asOfDate":"2026-06-01T00:00:00.000Z","basementEnclosureCrawlspaceType":null,"policyCount":1,"crsClassificationCode":null,"dateOfLoss":"1992-12-11T00:00:00.000Z","elevatedBuildingIndicator":false,"elev


### 7.2E Inspect NFIP Claims Fields

After confirming that the NFIP Redacted Claims endpoint is accessible, this step inspects the JSON structure and available fields.

The main purpose is to check whether the claims dataset contains the geography, date, and payment fields needed to create county-year flood-loss indicators.

In [78]:
# ==================================================
# 7.2E Inspect NFIP Claims Fields
# ==================================================

# --------------------------------------------------
# Convert claims response to JSON
# --------------------------------------------------

claims_json = claims_response.json()

# --------------------------------------------------
# Inspect top-level JSON keys
# --------------------------------------------------

print("Top-level JSON keys:")
print(claims_json.keys())

# --------------------------------------------------
# Inspect metadata
# --------------------------------------------------

print("\nMetadata:")
display(pd.DataFrame([claims_json["metadata"]]))

# --------------------------------------------------
# Extract records
# --------------------------------------------------

claims_records = claims_json["FimaNfipClaims"]

print("\nNumber of records returned:")
print(len(claims_records))

# --------------------------------------------------
# Convert sample records to dataframe
# --------------------------------------------------

claims_sample = pd.DataFrame(claims_records)

print("\nSample shape:")
print(claims_sample.shape)

print("\nAvailable columns:")
print(claims_sample.columns.tolist())

display(claims_sample.head())

Top-level JSON keys:
dict_keys(['metadata', 'FimaNfipClaims'])

Metadata:


,skip,select,rundate,top,filter,format,metadata,orderby,entityname,version,url,count
0,0,None,2026-06-22T04:52:32.510Z,5,,json,True,,FimaNfipClaims,v2,/api/open/v2/FimaNfipClaims?%24top=5,0



Number of records returned:
5

Sample shape:
(5, 73)

Available columns:
['agricultureStructureIndicator', 'asOfDate', 'basementEnclosureCrawlspaceType', 'policyCount', 'crsClassificationCode', 'dateOfLoss', 'elevatedBuildingIndicator', 'elevationCertificateIndicator', 'elevationDifference', 'baseFloodElevation', 'ratedFloodZone', 'houseWorship', 'locationOfContents', 'lowestAdjacentGrade', 'lowestFloorElevation', 'numberOfFloorsInTheInsuredBuilding', 'nonProfitIndicator', 'obstructionType', 'occupancyType', 'originalConstructionDate', 'originalNBDate', 'amountPaidOnBuildingClaim', 'amountPaidOnContentsClaim', 'amountPaidOnIncreasedCostOfComplianceClaim', 'postFIRMConstructionIndicator', 'rateMethod', 'smallBusinessIndicatorBuilding', 'totalBuildingInsuranceCoverage', 'totalContentsInsuranceCoverage', 'yearOfLoss', 'primaryResidenceIndicator', 'buildingDamageAmount', 'buildingDeductibleCode', 'netBuildingPaymentAmount', 'buildingPropertyValue', 'causeOfDamage', 'condominiumCoverageTyp

,agricultureStructureIndicator,asOfDate,basementEnclosureCrawlspaceType,policyCount,crsClassificationCode,dateOfLoss,elevatedBuildingIndicator,elevationCertificateIndicator,elevationDifference,baseFloodElevation,...,rentalPropertyIndicator,state,reportedCity,reportedZipCode,countyCode,censusTract,censusBlockGroupFips,latitude,longitude,id
0,False,2026-06-01T00:00:00.000Z,NaN,1,None,1992-12-11T00:00:00.000Z,False,NaN,NaN,NaN,...,False,NJ,Currently Unavailable,07732,34025,34025800100,340258001002,40.4,-74.0,3f994197-be45-40fd-8abf-9afc13b2e1ea
1,False,2026-06-01T00:00:00.000Z,NaN,1,None,2018-10-10T00:00:00.000Z,True,NaN,6.0,7.4,...,False,FL,Currently Unavailable,32328,12037,12037970304,120379703041,29.7,-84.9,856f876b-ef99-4bdd-92d3-ab2d95cde7f3
2,False,2026-06-01T00:00:00.000Z,2.0,1,None,1996-12-16T00:00:00.000Z,False,NaN,NaN,NaN,...,False,PA,Currently Unavailable,19403,42091,42091203402,420912034022,40.1,-75.4,157d2c40-4505-4792-a0f9-773c8f0bdc59
3,False,2026-06-01T00:00:00.000Z,NaN,1,None,2001-06-14T00:00:00.000Z,False,2,NaN,NaN,...,False,MS,Currently Unavailable,39466,28109,28109950700,281099507001,30.5,-89.7,1e47522a-f4c1-4f1c-9505-124a74c472e8
4,False,2026-06-01T00:00:00.000Z,NaN,1,None,1979-07-26T00:00:00.000Z,False,NaN,NaN,NaN,...,False,TX,Currently Unavailable,77511,48039,NaN,NaN,29.4,-95.2,e6a616fb-8cbd-4d19-b13a-f67b64dd60e4


### 7.2F Explore Florida NFIP Claims Coverage

After confirming that the NFIP Redacted Claims endpoint is accessible and contains historical loss-date fields, this step filters the claims data to Florida and the project period.

The purpose is to evaluate whether the claims dataset can be aggregated into county-year flood-insurance loss indicators for the vulnerability framework.

In [79]:
# ==================================================
# 7.2F Explore Florida NFIP Claims Coverage
# ==================================================

# --------------------------------------------------
# Query Florida NFIP claims during the project period
# --------------------------------------------------

claims_endpoint = "https://www.fema.gov/api/open/v2/FimaNfipClaims"

params = {
    "$filter": "state eq 'FL' and yearOfLoss ge 2011 and yearOfLoss le 2025",
    "$select": (
        "state,"
        "countyCode,"
        "yearOfLoss,"
        "dateOfLoss,"
        "amountPaidOnBuildingClaim,"
        "amountPaidOnContentsClaim,"
        "amountPaidOnIncreasedCostOfComplianceClaim,"
        "netBuildingPaymentAmount,"
        "netContentsPaymentAmount,"
        "netIccPaymentAmount,"
        "totalBuildingInsuranceCoverage,"
        "totalContentsInsuranceCoverage,"
        "policyCount,"
        "id"
    ),
    "$top": 5000
}

claims_fl_response = requests.get(
    claims_endpoint,
    params=params,
    timeout=120
)

print("Request URL:")
print(claims_fl_response.url)

print("\nStatus code:")
print(claims_fl_response.status_code)

print("\nResponse content type:")
print(claims_fl_response.headers.get("Content-Type"))

# --------------------------------------------------
# Convert response to JSON and extract records
# --------------------------------------------------

claims_fl_json = claims_fl_response.json()

print("\nTop-level JSON keys:")
print(claims_fl_json.keys())

claims_fl_records = claims_fl_json["FimaNfipClaims"]

claims_fl = pd.DataFrame(claims_fl_records)

print("\nFlorida NFIP claims sample shape:")
print(claims_fl.shape)

print("\nAvailable columns:")
print(claims_fl.columns.tolist())

display(claims_fl.head())

# --------------------------------------------------
# Basic coverage checks
# --------------------------------------------------

print("\nYear coverage:")
print(claims_fl["yearOfLoss"].min(), "to", claims_fl["yearOfLoss"].max())

print("\nClaims by year:")
display(
    claims_fl["yearOfLoss"]
    .value_counts()
    .sort_index()
    .reset_index()
    .rename(columns={"index": "yearOfLoss", "yearOfLoss": "claim_records"})
)

print("\nUnique county codes:")
print(claims_fl["countyCode"].nunique())

print("\nMissing countyCode values:")
print(claims_fl["countyCode"].isna().sum())

# --------------------------------------------------
# Check county FIPS match with project counties
# --------------------------------------------------

claims_fl["STCOFIPS"] = claims_fl["countyCode"].astype(str).str.zfill(5)

project_counties = set(county_year_acs_projected["STCOFIPS"].unique())
claims_counties = set(claims_fl["STCOFIPS"].dropna().unique())

missing_project_counties_from_claims = sorted(project_counties - claims_counties)
extra_claim_counties = sorted(claims_counties - project_counties)

print("\nProject counties:", len(project_counties))
print("NFIP claims counties:", len(claims_counties))
print("Matched counties:", len(project_counties & claims_counties))

print("\nProject counties with no NFIP claims in sampled query:")
print(missing_project_counties_from_claims)

print("\nExtra NFIP claim counties not in project dataset:")
print(extra_claim_counties)

Request URL:
https://www.fema.gov/api/open/v2/FimaNfipClaims?%24filter=state+eq+%27FL%27+and+yearOfLoss+ge+2011+and+yearOfLoss+le+2025&%24select=state%2CcountyCode%2CyearOfLoss%2CdateOfLoss%2CamountPaidOnBuildingClaim%2CamountPaidOnContentsClaim%2CamountPaidOnIncreasedCostOfComplianceClaim%2CnetBuildingPaymentAmount%2CnetContentsPaymentAmount%2CnetIccPaymentAmount%2CtotalBuildingInsuranceCoverage%2CtotalContentsInsuranceCoverage%2CpolicyCount%2Cid&%24top=5000

Status code:
200

Response content type:
application/json; charset=utf-8

Top-level JSON keys:
dict_keys(['metadata', 'FimaNfipClaims'])

Florida NFIP claims sample shape:
(5000, 14)

Available columns:
['state', 'countyCode', 'yearOfLoss', 'dateOfLoss', 'amountPaidOnBuildingClaim', 'amountPaidOnContentsClaim', 'amountPaidOnIncreasedCostOfComplianceClaim', 'netBuildingPaymentAmount', 'netContentsPaymentAmount', 'netIccPaymentAmount', 'totalBuildingInsuranceCoverage', 'totalContentsInsuranceCoverage', 'policyCount', 'id']


,state,countyCode,yearOfLoss,dateOfLoss,amountPaidOnBuildingClaim,amountPaidOnContentsClaim,amountPaidOnIncreasedCostOfComplianceClaim,netBuildingPaymentAmount,netContentsPaymentAmount,netIccPaymentAmount,totalBuildingInsuranceCoverage,totalContentsInsuranceCoverage,policyCount,id
0,FL,12037,2018,2018-10-10T00:00:00.000Z,39008.35,10512.11,0.0,39008.35,10512.11,0.0,250000,21000,1,856f876b-ef99-4bdd-92d3-ab2d95cde7f3
1,FL,12057,2024,2024-09-26T00:00:00.000Z,196036.65,43859.36,0.0,196036.65,43859.36,0.0,250000,100000,1,0c8f3c98-8827-4b96-b8af-1cf34a125ad7
2,FL,12021,2014,2014-08-05T00:00:00.000Z,4759.55,0.00,0.0,4759.55,0.00,0.0,250000,100000,1,e4b5ee8c-c86e-4e4b-b36c-d76be66c4a27
3,FL,12103,2024,2024-09-26T00:00:00.000Z,5761.43,0.00,0.0,5761.43,0.00,0.0,250000,100000,1,15d48095-1b1b-4743-b6bf-47dc2f33f0f9
4,FL,12127,2022,2022-09-29T00:00:00.000Z,176461.02,80000.00,0.0,176461.02,80000.00,0.0,250000,94000,1,d673e261-f248-4175-820b-19c6f7e84b50



Year coverage:
2011 to 2025

Claims by year:


,claim_records,count
0,2011,48
1,2012,137
2,2013,49
3,2014,86
4,2015,39
5,2016,197
6,2017,763
7,2018,107
8,2019,23
9,2020,227



Unique county codes:
56

Missing countyCode values:
3

Project counties: 67
NFIP claims counties: 56
Matched counties: 56

Project counties with no NFIP claims in sampled query:
['12007', '12013', '12039', '12043', '12047', '12051', '12065', '12067', '12077', '12125', '12133']

Extra NFIP claim counties not in project dataset:
[]


### 7.2G Retrieve All Florida NFIP Claims for the Project Period

The first NFIP claims query returned 5,000 records, which was exactly the requested limit. This suggests that additional Florida claim records may exist beyond the first page of results.

This step retrieves all available Florida NFIP claim records from 2011 to 2025 using pagination. Only selected fields needed for county-year aggregation are requested to keep the query focused and manageable.

In [80]:
# ==================================================
# 7.2G Retrieve All Florida NFIP Claims for the Project Period
# ==================================================

# --------------------------------------------------
# Define endpoint and selected fields
# --------------------------------------------------

claims_endpoint = "https://www.fema.gov/api/open/v2/FimaNfipClaims"

claims_select_cols = [
    "state",
    "countyCode",
    "yearOfLoss",
    "dateOfLoss",
    "amountPaidOnBuildingClaim",
    "amountPaidOnContentsClaim",
    "amountPaidOnIncreasedCostOfComplianceClaim",
    "netBuildingPaymentAmount",
    "netContentsPaymentAmount",
    "netIccPaymentAmount",
    "totalBuildingInsuranceCoverage",
    "totalContentsInsuranceCoverage",
    "policyCount",
    "id"
]

claims_filter = "state eq 'FL' and yearOfLoss ge 2011 and yearOfLoss le 2025"

# --------------------------------------------------
# Retrieve records using pagination
# --------------------------------------------------

all_claim_records = []

batch_size = 5000
skip = 0

while True:
    
    params = {
        "$filter": claims_filter,
        "$select": ",".join(claims_select_cols),
        "$top": batch_size,
        "$skip": skip
    }
    
    response = requests.get(
        claims_endpoint,
        params=params,
        timeout=120
    )
    
    print(f"Requested records {skip} to {skip + batch_size}")
    print("Status code:", response.status_code)
    
    if response.status_code != 200:
        print("Request failed.")
        print(response.text[:500])
        break
    
    claims_json = response.json()
    batch_records = claims_json["FimaNfipClaims"]
    
    print("Records returned:", len(batch_records))
    
    if len(batch_records) == 0:
        break
    
    all_claim_records.extend(batch_records)
    
    if len(batch_records) < batch_size:
        break
    
    skip += batch_size

# --------------------------------------------------
# Convert all retrieved records to dataframe
# --------------------------------------------------

claims_fl_all = pd.DataFrame(all_claim_records)

print("\nTotal Florida NFIP claim records retrieved:")
print(len(claims_fl_all))

print("\nDataset shape:")
print(claims_fl_all.shape)

display(claims_fl_all.head())

Requested records 0 to 5000
Status code: 200
Records returned: 5000
Requested records 5000 to 10000
Status code: 200
Records returned: 5000
Requested records 10000 to 15000
Status code: 200
Records returned: 5000
Requested records 15000 to 20000
Status code: 200
Records returned: 5000
Requested records 20000 to 25000
Status code: 200
Records returned: 5000
Requested records 25000 to 30000
Status code: 200
Records returned: 5000
Requested records 30000 to 35000
Status code: 200
Records returned: 5000
Requested records 35000 to 40000
Status code: 200
Records returned: 5000
Requested records 40000 to 45000
Status code: 200
Records returned: 5000
Requested records 45000 to 50000
Status code: 200
Records returned: 5000
Requested records 50000 to 55000
Status code: 200
Records returned: 5000
Requested records 55000 to 60000
Status code: 200
Records returned: 5000
Requested records 60000 to 65000
Status code: 200
Records returned: 5000
Requested records 65000 to 70000
Status code: 200
Records

,state,countyCode,yearOfLoss,dateOfLoss,amountPaidOnBuildingClaim,amountPaidOnContentsClaim,amountPaidOnIncreasedCostOfComplianceClaim,netBuildingPaymentAmount,netContentsPaymentAmount,netIccPaymentAmount,totalBuildingInsuranceCoverage,totalContentsInsuranceCoverage,policyCount,id
0,FL,12037,2018,2018-10-10T00:00:00.000Z,39008.35,10512.11,0.0,39008.35,10512.11,0.0,250000,21000.0,1,856f876b-ef99-4bdd-92d3-ab2d95cde7f3
1,FL,12057,2024,2024-09-26T00:00:00.000Z,196036.65,43859.36,0.0,196036.65,43859.36,0.0,250000,100000.0,1,0c8f3c98-8827-4b96-b8af-1cf34a125ad7
2,FL,12021,2014,2014-08-05T00:00:00.000Z,4759.55,0.00,0.0,4759.55,0.00,0.0,250000,100000.0,1,e4b5ee8c-c86e-4e4b-b36c-d76be66c4a27
3,FL,12103,2024,2024-09-26T00:00:00.000Z,5761.43,0.00,0.0,5761.43,0.00,0.0,250000,100000.0,1,15d48095-1b1b-4743-b6bf-47dc2f33f0f9
4,FL,12127,2022,2022-09-29T00:00:00.000Z,176461.02,80000.00,0.0,176461.02,80000.00,0.0,250000,94000.0,1,d673e261-f248-4175-820b-19c6f7e84b50


### 7.2H Validate Florida NFIP Claims Pull

After retrieving all Florida NFIP claims for the project period, this step validates the raw claims pull before any aggregation.

The purpose is to confirm that the claims dataset covers the correct years, contains Florida-only records, includes usable county identifiers, and has the payment fields needed to create county-year flood-loss indicators.

In [81]:
# ==================================================
# 7.2H Validate Florida NFIP Claims Pull
# ==================================================

# --------------------------------------------------
# Basic dataset checks
# --------------------------------------------------

print("Florida NFIP claims dataset shape:")
print(claims_fl_all.shape)

print("\nUnique states:")
print(claims_fl_all["state"].unique())

print("\nYear coverage:")
print(claims_fl_all["yearOfLoss"].min(), "to", claims_fl_all["yearOfLoss"].max())

print("\nUnique claim IDs:")
print(claims_fl_all["id"].nunique())

print("\nDuplicate claim IDs:")
print(claims_fl_all["id"].duplicated().sum())

# --------------------------------------------------
# County coverage checks
# --------------------------------------------------

print("\nMissing countyCode values:")
print(claims_fl_all["countyCode"].isna().sum())

claims_fl_all["STCOFIPS"] = claims_fl_all["countyCode"].astype(str).str.zfill(5)

project_counties = set(county_year_acs_projected["STCOFIPS"].unique())
claims_counties = set(claims_fl_all["STCOFIPS"].dropna().unique())

missing_project_counties_from_claims = sorted(project_counties - claims_counties)
extra_claim_counties = sorted(claims_counties - project_counties)

print("\nProject counties:", len(project_counties))
print("NFIP claims counties:", len(claims_counties))
print("Matched counties:", len(project_counties & claims_counties))

print("\nProject counties with no NFIP claims in full claims pull:")
print(missing_project_counties_from_claims)

print("\nExtra NFIP claim counties not in project dataset:")
print(extra_claim_counties)

# --------------------------------------------------
# Claims by year
# --------------------------------------------------

claims_by_year = (
    claims_fl_all
    .groupby("yearOfLoss")
    .size()
    .reset_index(name="claim_records")
    .sort_values("yearOfLoss")
)

print("\nClaims by year:")
display(claims_by_year)

# --------------------------------------------------
# Missing values in key claims fields
# --------------------------------------------------

claims_key_cols = [
    "countyCode",
    "yearOfLoss",
    "dateOfLoss",
    "amountPaidOnBuildingClaim",
    "amountPaidOnContentsClaim",
    "amountPaidOnIncreasedCostOfComplianceClaim",
    "netBuildingPaymentAmount",
    "netContentsPaymentAmount",
    "netIccPaymentAmount",
    "policyCount"
]

print("\nMissing values in key claims fields:")
print(claims_fl_all[claims_key_cols].isna().sum())

# --------------------------------------------------
# Payment field summaries
# --------------------------------------------------

payment_cols = [
    "amountPaidOnBuildingClaim",
    "amountPaidOnContentsClaim",
    "amountPaidOnIncreasedCostOfComplianceClaim",
    "netBuildingPaymentAmount",
    "netContentsPaymentAmount",
    "netIccPaymentAmount"
]

print("\nPayment field summary:")
display(claims_fl_all[payment_cols].describe().round(2))

Florida NFIP claims dataset shape:
(214827, 14)

Unique states:
<StringArray>
['FL']
Length: 1, dtype: str

Year coverage:
2011 to 2025

Unique claim IDs:
214827

Duplicate claim IDs:
0

Missing countyCode values:
85

Project counties: 67
NFIP claims counties: 67
Matched counties: 67

Project counties with no NFIP claims in full claims pull:
[]

Extra NFIP claim counties not in project dataset:
[]

Claims by year:


,yearOfLoss,claim_records
0,2011,2157
1,2012,5369
2,2013,2154
3,2014,3960
4,2015,1993
5,2016,8838
6,2017,31430
7,2018,5100
8,2019,1193
9,2020,9992



Missing values in key claims fields:
countyCode                                       85
yearOfLoss                                        0
dateOfLoss                                        0
amountPaidOnBuildingClaim                     31476
amountPaidOnContentsClaim                     31476
amountPaidOnIncreasedCostOfComplianceClaim    31476
netBuildingPaymentAmount                          0
netContentsPaymentAmount                          0
netIccPaymentAmount                               0
policyCount                                       0
dtype: int64

Payment field summary:


,amountPaidOnBuildingClaim,amountPaidOnContentsClaim,amountPaidOnIncreasedCostOfComplianceClaim,netBuildingPaymentAmount,netContentsPaymentAmount,netIccPaymentAmount
count,183351.00,183351.00,183351.00,214827.00,214827.00,214827.00
mean,75374.51,10267.18,141.97,64289.69,8759.23,120.94
std,160546.87,24108.79,2202.11,150665.96,22565.60,2031.27
min,-201667.50,-80000.00,-6450.00,-201667.50,-80000.00,-6450.00
25%,5573.34,0.00,0.00,0.00,0.00,0.00
50%,35795.38,0.00,0.00,20000.00,0.00,0.00
75%,101961.90,10000.00,0.00,89622.07,6304.16,0.00
max,10741476.93,500000.00,468074.68,10741476.93,500000.00,468074.68


#### 7.2H Output Interpretation

The full Florida NFIP claims pull was successfully retrieved for the 2011–2025 project period.

The dataset contains 214,827 Florida NFIP claim records, covering all years from 2011 to 2025. All records are from Florida, and there are no duplicate claim IDs.

County coverage is strong. All 67 Florida counties in the project dataset are represented in the NFIP claims data, and no extra counties outside the project dataset were found. This confirms that the claims dataset can be matched to the county-year vulnerability dataset using `countyCode`.

Only 85 records have missing county codes. Because county identifiers are required for county-year aggregation, these records will be excluded from the aggregated NFIP feature layer.

The claims dataset includes complete net payment fields, including `netBuildingPaymentAmount`, `netContentsPaymentAmount`, and `netIccPaymentAmount`. These fields will be used to calculate total NFIP claim payments because they have no missing values. Some payment values are negative, which may reflect transaction adjustments or corrections in the insurance records. These values are retained because they are part of the reported net payment data.

Overall, the NFIP claims dataset is feasible for integration and can support county-year flood-insurance loss indicators such as claim counts, total claim payments, average claim payments, cumulative claim burden, and recent three-year claim burden.

### 7.3 Create County-Year NFIP Claims Indicators

After confirming that the NFIP Redacted Claims dataset is feasible for integration, this step creates county-year insurance-loss indicators.

The raw claims dataset contains individual NFIP claim records. To align it with the project dataset, the claims are aggregated to the county-year level using `countyCode` and `yearOfLoss`.

This step creates annual NFIP indicators such as claim counts, total claim payments, average claim payments, policy counts, coverage amounts, cumulative claim burden, and recent three-year claim burden. A complete county-year grid is also created so that county-years with no NFIP claims are retained and assigned zero values.

Although the full county-year grid contains 1,005 rows, the current project dataset contains 1,000 county-year observations because Monroe County is not available for 2011–2015 in the cleaned housing dataset.

To avoid introducing county-year rows that do not exist in the main modeling dataset, the NFIP claims indicators are aligned to the exact county-year structure of the ACS-enhanced project dataset.

In [82]:
# ==================================================
# 7.3 Create County-Year NFIP Claims Indicators
# ==================================================

# --------------------------------------------------
# Create a working copy of the full Florida NFIP claims dataset
# --------------------------------------------------

claims_clean = claims_fl_all.copy()

print("Raw Florida NFIP claims shape:")
print(claims_clean.shape)

# --------------------------------------------------
# Drop claims with missing countyCode
# --------------------------------------------------

missing_county_records = claims_clean["countyCode"].isna().sum()

claims_clean = claims_clean.dropna(subset=["countyCode"]).copy()

print("\nRecords dropped due to missing countyCode:")
print(missing_county_records)

print("\nClaims shape after dropping missing countyCode:")
print(claims_clean.shape)

# --------------------------------------------------
# Prepare county and year identifiers
# --------------------------------------------------

claims_clean["STCOFIPS"] = (
    claims_clean["countyCode"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.zfill(5)
)

claims_clean["Year"] = claims_clean["yearOfLoss"].astype(int)

# --------------------------------------------------
# Convert payment and coverage fields to numeric format
# --------------------------------------------------

claims_numeric_cols = [
    "netBuildingPaymentAmount",
    "netContentsPaymentAmount",
    "netIccPaymentAmount",
    "totalBuildingInsuranceCoverage",
    "totalContentsInsuranceCoverage",
    "policyCount"
]

for col in claims_numeric_cols:
    claims_clean[col] = pd.to_numeric(claims_clean[col], errors="coerce")

# --------------------------------------------------
# Create total NFIP claim payment
# --------------------------------------------------

claims_clean["nfip_total_claim_payment"] = (
    claims_clean["netBuildingPaymentAmount"].fillna(0)
    + claims_clean["netContentsPaymentAmount"].fillna(0)
    + claims_clean["netIccPaymentAmount"].fillna(0)
)

# --------------------------------------------------
# Aggregate claims to county-year level
# --------------------------------------------------

nfip_county_year = (
    claims_clean
    .groupby(["STCOFIPS", "Year"])
    .agg(
        nfip_claim_count=("id", "count"),
        nfip_policy_count_sum=("policyCount", "sum"),
        nfip_total_building_payment=("netBuildingPaymentAmount", "sum"),
        nfip_total_contents_payment=("netContentsPaymentAmount", "sum"),
        nfip_total_icc_payment=("netIccPaymentAmount", "sum"),
        nfip_total_claim_payment=("nfip_total_claim_payment", "sum"),
        nfip_avg_claim_payment=("nfip_total_claim_payment", "mean"),
        nfip_total_building_coverage=("totalBuildingInsuranceCoverage", "sum"),
        nfip_total_contents_coverage=("totalContentsInsuranceCoverage", "sum")
    )
    .reset_index()
)

print("\nAggregated NFIP county-year shape:")
print(nfip_county_year.shape)

display(nfip_county_year.head())

# --------------------------------------------------
# Create complete county-year grid for project counties and years
# --------------------------------------------------

project_counties = sorted(county_year_acs_projected["STCOFIPS"].unique())
project_years = sorted(county_year_acs_projected["Year"].unique())

complete_county_year_grid = pd.MultiIndex.from_product(
    [project_counties, project_years],
    names=["STCOFIPS", "Year"]
).to_frame(index=False)

print("\nComplete county-year grid shape:")
print(complete_county_year_grid.shape)

# --------------------------------------------------
# Merge aggregated NFIP claims onto complete county-year grid
# --------------------------------------------------

nfip_county_year_complete = complete_county_year_grid.merge(
    nfip_county_year,
    on=["STCOFIPS", "Year"],
    how="left"
)

# --------------------------------------------------
# Fill no-claim county-years with zero
# --------------------------------------------------

nfip_fill_zero_cols = [
    "nfip_claim_count",
    "nfip_policy_count_sum",
    "nfip_total_building_payment",
    "nfip_total_contents_payment",
    "nfip_total_icc_payment",
    "nfip_total_claim_payment",
    "nfip_avg_claim_payment",
    "nfip_total_building_coverage",
    "nfip_total_contents_coverage"
]

nfip_county_year_complete[nfip_fill_zero_cols] = (
    nfip_county_year_complete[nfip_fill_zero_cols]
    .fillna(0)
)

# --------------------------------------------------
# Create cumulative NFIP indicators
# --------------------------------------------------

nfip_county_year_complete = nfip_county_year_complete.sort_values(
    ["STCOFIPS", "Year"]
).reset_index(drop=True)

nfip_county_year_complete["nfip_cumulative_claim_count"] = (
    nfip_county_year_complete
    .groupby("STCOFIPS")["nfip_claim_count"]
    .cumsum()
)

nfip_county_year_complete["nfip_cumulative_claim_payment"] = (
    nfip_county_year_complete
    .groupby("STCOFIPS")["nfip_total_claim_payment"]
    .cumsum()
)

# --------------------------------------------------
# Create recent 3-year NFIP indicators
# --------------------------------------------------

nfip_county_year_complete["nfip_recent_3yr_claim_count"] = (
    nfip_county_year_complete
    .groupby("STCOFIPS")["nfip_claim_count"]
    .rolling(window=3, min_periods=1)
    .sum()
    .reset_index(level=0, drop=True)
)

nfip_county_year_complete["nfip_recent_3yr_claim_payment"] = (
    nfip_county_year_complete
    .groupby("STCOFIPS")["nfip_total_claim_payment"]
    .rolling(window=3, min_periods=1)
    .sum()
    .reset_index(level=0, drop=True)
)

# --------------------------------------------------
# Create claim occurrence indicator
# --------------------------------------------------

nfip_county_year_complete["nfip_claim_year_indicator"] = (
    nfip_county_year_complete["nfip_claim_count"] > 0
).astype(int)

# --------------------------------------------------
# Final validation preview
# --------------------------------------------------

print("\nComplete NFIP county-year feature dataset shape:")
print(nfip_county_year_complete.shape)

print("\nUnique counties:")
print(nfip_county_year_complete["STCOFIPS"].nunique())

print("\nYear range:")
print(
    nfip_county_year_complete["Year"].min(),
    "to",
    nfip_county_year_complete["Year"].max()
)

print("\nMissing values in NFIP county-year features:")
print(nfip_county_year_complete.isna().sum())

print("\nSummary of key NFIP indicators:")
display(
    nfip_county_year_complete[
        [
            "nfip_claim_count",
            "nfip_total_claim_payment",
            "nfip_avg_claim_payment",
            "nfip_cumulative_claim_count",
            "nfip_cumulative_claim_payment",
            "nfip_recent_3yr_claim_count",
            "nfip_recent_3yr_claim_payment",
            "nfip_claim_year_indicator"
        ]
    ].describe().round(2)
)

display(nfip_county_year_complete.head())

Raw Florida NFIP claims shape:
(214827, 15)

Records dropped due to missing countyCode:
85

Claims shape after dropping missing countyCode:
(214742, 15)

Aggregated NFIP county-year shape:
(764, 11)


,STCOFIPS,Year,nfip_claim_count,nfip_policy_count_sum,nfip_total_building_payment,nfip_total_contents_payment,nfip_total_icc_payment,nfip_total_claim_payment,nfip_avg_claim_payment,nfip_total_building_coverage,nfip_total_contents_coverage
0,12001,2012,10,10,200195.98,0.00,0.0,200195.98,20019.598000,1767900,448500.0
1,12001,2013,3,3,5630.43,0.00,0.0,5630.43,1876.810000,376000,132100.0
2,12001,2014,2,2,12849.49,0.00,0.0,12849.49,6424.745000,500000,200000.0
3,12001,2015,6,6,129034.12,4629.94,0.0,133664.06,22277.343333,1172900,334500.0
4,12001,2016,1,1,0.00,0.00,0.0,0.00,0.000000,250000,100000.0



Complete county-year grid shape:
(1005, 2)

Complete NFIP county-year feature dataset shape:
(1005, 16)

Unique counties:
67

Year range:
2011 to 2025

Missing values in NFIP county-year features:
STCOFIPS                         0
Year                             0
nfip_claim_count                 0
nfip_policy_count_sum            0
nfip_total_building_payment      0
nfip_total_contents_payment      0
nfip_total_icc_payment           0
nfip_total_claim_payment         0
nfip_avg_claim_payment           0
nfip_total_building_coverage     0
nfip_total_contents_coverage     0
nfip_cumulative_claim_count      0
nfip_cumulative_claim_payment    0
nfip_recent_3yr_claim_count      0
nfip_recent_3yr_claim_payment    0
nfip_claim_year_indicator        0
dtype: int64

Summary of key NFIP indicators:


,nfip_claim_count,nfip_total_claim_payment,nfip_avg_claim_payment,nfip_cumulative_claim_count,nfip_cumulative_claim_payment,nfip_recent_3yr_claim_count,nfip_recent_3yr_claim_payment,nfip_claim_year_indicator
count,1005.00,1.005000e+03,1005.00,1005.00,1.005000e+03,1005.00,1.005000e+03,1005.00
mean,213.67,1.563736e+07,12665.21,1077.47,5.729651e+07,559.59,3.904378e+07,0.76
std,1441.38,1.624602e+08,21662.53,3242.69,3.005773e+08,2446.80,2.648696e+08,0.43
min,0.00,0.000000e+00,0.00,0.00,0.000000e+00,0.00,0.000000e+00,0.00
25%,1.00,0.000000e+00,0.00,14.00,1.710510e+05,5.00,2.676706e+04,1.00
50%,5.00,2.917229e+04,3851.96,108.00,2.371942e+06,36.00,3.881160e+05,1.00
75%,33.00,3.524594e+05,15945.17,670.00,1.483198e+07,223.00,4.328379e+06,1.00
max,28853.00,3.388301e+09,288747.17,39411.00,3.878872e+09,36325.00,3.808979e+09,1.00


,STCOFIPS,Year,nfip_claim_count,nfip_policy_count_sum,nfip_total_building_payment,nfip_total_contents_payment,nfip_total_icc_payment,nfip_total_claim_payment,nfip_avg_claim_payment,nfip_total_building_coverage,nfip_total_contents_coverage,nfip_cumulative_claim_count,nfip_cumulative_claim_payment,nfip_recent_3yr_claim_count,nfip_recent_3yr_claim_payment,nfip_claim_year_indicator
0,12001,2011,0.0,0.0,0.00,0.00,0.0,0.00,0.000000,0.0,0.0,0.0,0.00,0.0,0.00,0
1,12001,2012,10.0,10.0,200195.98,0.00,0.0,200195.98,20019.598000,1767900.0,448500.0,10.0,200195.98,10.0,200195.98,1
2,12001,2013,3.0,3.0,5630.43,0.00,0.0,5630.43,1876.810000,376000.0,132100.0,13.0,205826.41,13.0,205826.41,1
3,12001,2014,2.0,2.0,12849.49,0.00,0.0,12849.49,6424.745000,500000.0,200000.0,15.0,218675.90,15.0,218675.90,1
4,12001,2015,6.0,6.0,129034.12,4629.94,0.0,133664.06,22277.343333,1172900.0,334500.0,21.0,352339.96,11.0,152143.98,1


In [83]:
# ==================================================
# 7.3B Align NFIP Claims Indicators with Project County-Year Structure
# ==================================================

# --------------------------------------------------
# Create project-aligned county-year grid
# --------------------------------------------------

project_county_year_grid = (
    county_year_acs_projected[["STCOFIPS", "Year"]]
    .drop_duplicates()
    .copy()
)

project_county_year_grid["STCOFIPS"] = (
    project_county_year_grid["STCOFIPS"]
    .astype(str)
    .str.zfill(5)
)

project_county_year_grid["Year"] = project_county_year_grid["Year"].astype(int)

print("Project county-year grid shape:")
print(project_county_year_grid.shape)

print("\nUnique counties:")
print(project_county_year_grid["STCOFIPS"].nunique())

print("\nYear range:")
print(
    project_county_year_grid["Year"].min(),
    "to",
    project_county_year_grid["Year"].max()
)

# --------------------------------------------------
# Merge aggregated NFIP claims onto project county-year grid
# --------------------------------------------------

nfip_county_year_project = project_county_year_grid.merge(
    nfip_county_year,
    on=["STCOFIPS", "Year"],
    how="left"
)

# --------------------------------------------------
# Fill no-claim county-years with zero
# --------------------------------------------------

nfip_fill_zero_cols = [
    "nfip_claim_count",
    "nfip_policy_count_sum",
    "nfip_total_building_payment",
    "nfip_total_contents_payment",
    "nfip_total_icc_payment",
    "nfip_total_claim_payment",
    "nfip_avg_claim_payment",
    "nfip_total_building_coverage",
    "nfip_total_contents_coverage"
]

nfip_county_year_project[nfip_fill_zero_cols] = (
    nfip_county_year_project[nfip_fill_zero_cols]
    .fillna(0)
)

# --------------------------------------------------
# Sort before cumulative and rolling indicators
# --------------------------------------------------

nfip_county_year_project = nfip_county_year_project.sort_values(
    ["STCOFIPS", "Year"]
).reset_index(drop=True)

# --------------------------------------------------
# Create cumulative NFIP indicators
# --------------------------------------------------

nfip_county_year_project["nfip_cumulative_claim_count"] = (
    nfip_county_year_project
    .groupby("STCOFIPS")["nfip_claim_count"]
    .cumsum()
)

nfip_county_year_project["nfip_cumulative_claim_payment"] = (
    nfip_county_year_project
    .groupby("STCOFIPS")["nfip_total_claim_payment"]
    .cumsum()
)

# --------------------------------------------------
# Create recent 3-year NFIP indicators
# --------------------------------------------------

nfip_county_year_project["nfip_recent_3yr_claim_count"] = (
    nfip_county_year_project
    .groupby("STCOFIPS")["nfip_claim_count"]
    .rolling(window=3, min_periods=1)
    .sum()
    .reset_index(level=0, drop=True)
)

nfip_county_year_project["nfip_recent_3yr_claim_payment"] = (
    nfip_county_year_project
    .groupby("STCOFIPS")["nfip_total_claim_payment"]
    .rolling(window=3, min_periods=1)
    .sum()
    .reset_index(level=0, drop=True)
)

# --------------------------------------------------
# Create claim occurrence indicator
# --------------------------------------------------

nfip_county_year_project["nfip_claim_year_indicator"] = (
    nfip_county_year_project["nfip_claim_count"] > 0
).astype(int)

# --------------------------------------------------
# Final validation
# --------------------------------------------------

print("\nProject-aligned NFIP county-year feature dataset shape:")
print(nfip_county_year_project.shape)

print("\nUnique counties:")
print(nfip_county_year_project["STCOFIPS"].nunique())

print("\nYear range:")
print(
    nfip_county_year_project["Year"].min(),
    "to",
    nfip_county_year_project["Year"].max()
)

print("\nMissing values in project-aligned NFIP features:")
print(nfip_county_year_project.isna().sum())

print("\nDuplicate county-year rows:")
print(
    nfip_county_year_project
    .duplicated(subset=["STCOFIPS", "Year"])
    .sum()
)

print("\nSummary of key NFIP indicators:")
display(
    nfip_county_year_project[
        [
            "nfip_claim_count",
            "nfip_total_claim_payment",
            "nfip_avg_claim_payment",
            "nfip_cumulative_claim_count",
            "nfip_cumulative_claim_payment",
            "nfip_recent_3yr_claim_count",
            "nfip_recent_3yr_claim_payment",
            "nfip_claim_year_indicator"
        ]
    ].describe().round(2)
)

display(nfip_county_year_project.head())

Project county-year grid shape:
(1000, 2)

Unique counties:
67

Year range:
2011 to 2025

Project-aligned NFIP county-year feature dataset shape:
(1000, 16)

Unique counties:
67

Year range:
2011 to 2025

Missing values in project-aligned NFIP features:
STCOFIPS                         0
Year                             0
nfip_claim_count                 0
nfip_policy_count_sum            0
nfip_total_building_payment      0
nfip_total_contents_payment      0
nfip_total_icc_payment           0
nfip_total_claim_payment         0
nfip_avg_claim_payment           0
nfip_total_building_coverage     0
nfip_total_contents_coverage     0
nfip_cumulative_claim_count      0
nfip_cumulative_claim_payment    0
nfip_recent_3yr_claim_count      0
nfip_recent_3yr_claim_payment    0
nfip_claim_year_indicator        0
dtype: int64

Duplicate county-year rows:
0

Summary of key NFIP indicators:


,nfip_claim_count,nfip_total_claim_payment,nfip_avg_claim_payment,nfip_cumulative_claim_count,nfip_cumulative_claim_payment,nfip_recent_3yr_claim_count,nfip_recent_3yr_claim_payment,nfip_claim_year_indicator
count,1000.00,1.000000e+03,1000.00,1000.00,1.000000e+03,1000.00,1.000000e+03,1000.00
mean,214.65,1.571465e+07,12677.02,1081.57,5.757097e+07,562.10,3.923631e+07,0.76
std,1444.92,1.628626e+08,21712.11,3248.08,3.012952e+08,2452.63,2.655175e+08,0.43
min,0.00,0.000000e+00,0.00,0.00,0.000000e+00,0.00,0.000000e+00,0.00
25%,1.00,0.000000e+00,0.00,13.75,1.695604e+05,4.00,2.426701e+04,1.00
50%,5.00,2.859842e+04,3833.38,109.50,2.377158e+06,36.00,3.880037e+05,1.00
75%,33.25,3.654569e+05,15914.39,673.25,1.488534e+07,225.25,4.372397e+06,1.00
max,28853.00,3.388301e+09,288747.17,39411.00,3.878872e+09,36325.00,3.808979e+09,1.00


,STCOFIPS,Year,nfip_claim_count,nfip_policy_count_sum,nfip_total_building_payment,nfip_total_contents_payment,nfip_total_icc_payment,nfip_total_claim_payment,nfip_avg_claim_payment,nfip_total_building_coverage,nfip_total_contents_coverage,nfip_cumulative_claim_count,nfip_cumulative_claim_payment,nfip_recent_3yr_claim_count,nfip_recent_3yr_claim_payment,nfip_claim_year_indicator
0,12001,2011,0.0,0.0,0.00,0.00,0.0,0.00,0.000000,0.0,0.0,0.0,0.00,0.0,0.00,0
1,12001,2012,10.0,10.0,200195.98,0.00,0.0,200195.98,20019.598000,1767900.0,448500.0,10.0,200195.98,10.0,200195.98,1
2,12001,2013,3.0,3.0,5630.43,0.00,0.0,5630.43,1876.810000,376000.0,132100.0,13.0,205826.41,13.0,205826.41,1
3,12001,2014,2.0,2.0,12849.49,0.00,0.0,12849.49,6424.745000,500000.0,200000.0,15.0,218675.90,15.0,218675.90,1
4,12001,2015,6.0,6.0,129034.12,4629.94,0.0,133664.06,22277.343333,1172900.0,334500.0,21.0,352339.96,11.0,152143.98,1


#### 7.3B Output Interpretation

The NFIP claims indicators were successfully aligned with the exact county-year structure of the project dataset.

The project-aligned NFIP feature dataset contains 1,000 county-year observations, 67 Florida counties, and covers the full 2011–2025 study period. There are no missing values and no duplicate county-year records, confirming that the NFIP feature layer is ready to merge with the ACS-enhanced dataset.

The summary statistics show that NFIP claim activity is highly skewed. Most county-years have relatively low claim counts and claim payments, while a small number of disaster-affected county-years show extremely large claim counts and payments. This pattern is expected for flood-insurance data and supports the use of both annual, cumulative, and recent three-year claim indicators.

The annual indicators capture current-year flood-insurance activity, while the cumulative indicators capture long-term repeated-loss burden. The recent three-year indicators provide a short-term measure of recent flood-insurance stress. Together, these variables add an important insurance-loss and financial-stress layer to the climate-housing vulnerability framework.

### 7.4 Merge NFIP Claims Indicators with ACS-Enhanced Dataset

After creating the project-aligned NFIP county-year claims indicators, this step merges the NFIP insurance-loss features with the ACS-enhanced county-year dataset.

The merge is performed using `STCOFIPS` and `Year`, which uniquely identify each county-year observation. Since the NFIP feature layer was already aligned to the same county-year structure as the main project dataset, the merge should preserve the number of rows and counties in the dataset.

In [84]:
# ==================================================
# 7.4 Merge NFIP Claims Indicators with ACS-Enhanced Dataset
# ==================================================

# --------------------------------------------------
# Create working copies
# --------------------------------------------------

county_year_base = county_year_acs_projected.copy()
nfip_features = nfip_county_year_project.copy()

print("ACS-enhanced county-year dataset shape:")
print(county_year_base.shape)

print("\nNFIP feature dataset shape:")
print(nfip_features.shape)

# --------------------------------------------------
# Ensure merge keys are consistently formatted
# --------------------------------------------------

county_year_base["STCOFIPS"] = (
    county_year_base["STCOFIPS"]
    .astype(str)
    .str.zfill(5)
)

nfip_features["STCOFIPS"] = (
    nfip_features["STCOFIPS"]
    .astype(str)
    .str.zfill(5)
)

county_year_base["Year"] = county_year_base["Year"].astype(int)
nfip_features["Year"] = nfip_features["Year"].astype(int)

# --------------------------------------------------
# Select NFIP feature columns for merge
# --------------------------------------------------

nfip_merge_cols = [
    "STCOFIPS",
    "Year",
    "nfip_claim_count",
    "nfip_policy_count_sum",
    "nfip_total_building_payment",
    "nfip_total_contents_payment",
    "nfip_total_icc_payment",
    "nfip_total_claim_payment",
    "nfip_avg_claim_payment",
    "nfip_total_building_coverage",
    "nfip_total_contents_coverage",
    "nfip_cumulative_claim_count",
    "nfip_cumulative_claim_payment",
    "nfip_recent_3yr_claim_count",
    "nfip_recent_3yr_claim_payment",
    "nfip_claim_year_indicator"
]

nfip_features_for_merge = nfip_features[nfip_merge_cols].copy()

# --------------------------------------------------
# Merge NFIP features with ACS-enhanced dataset
# --------------------------------------------------

county_year_nfip = county_year_base.merge(
    nfip_features_for_merge,
    on=["STCOFIPS", "Year"],
    how="left"
)

print("\nNFIP-enhanced county-year dataset shape:")
print(county_year_nfip.shape)

print("\nUnique counties:")
print(county_year_nfip["STCOFIPS"].nunique())

print("\nYear range:")
print(
    county_year_nfip["Year"].min(),
    "to",
    county_year_nfip["Year"].max()
)

# --------------------------------------------------
# Validate merge
# --------------------------------------------------

print("\nRows before merge:")
print(len(county_year_base))

print("\nRows after merge:")
print(len(county_year_nfip))

print("\nDuplicate county-year rows after merge:")
print(
    county_year_nfip
    .duplicated(subset=["STCOFIPS", "Year"])
    .sum()
)

print("\nMissing values in merged NFIP features:")
print(county_year_nfip[nfip_merge_cols[2:]].isna().sum())

# --------------------------------------------------
# Preview merged dataset
# --------------------------------------------------

preview_cols = [
    "STCOFIPS",
    "Year",
    "avg_annual_housing_price",
    "median_household_income",
    "price_to_income_ratio",
    "total_disaster_declarations",
    "nfip_claim_count",
    "nfip_total_claim_payment",
    "nfip_cumulative_claim_payment",
    "nfip_recent_3yr_claim_payment",
    "nfip_claim_year_indicator"
]

display(county_year_nfip[preview_cols].head())

ACS-enhanced county-year dataset shape:
(1000, 61)

NFIP feature dataset shape:
(1000, 16)

NFIP-enhanced county-year dataset shape:
(1000, 75)

Unique counties:
67

Year range:
2011 to 2025

Rows before merge:
1000

Rows after merge:
1000

Duplicate county-year rows after merge:
0

Missing values in merged NFIP features:
nfip_claim_count                 0
nfip_policy_count_sum            0
nfip_total_building_payment      0
nfip_total_contents_payment      0
nfip_total_icc_payment           0
nfip_total_claim_payment         0
nfip_avg_claim_payment           0
nfip_total_building_coverage     0
nfip_total_contents_coverage     0
nfip_cumulative_claim_count      0
nfip_cumulative_claim_payment    0
nfip_recent_3yr_claim_count      0
nfip_recent_3yr_claim_payment    0
nfip_claim_year_indicator        0
dtype: int64


KeyError: "['total_disaster_declarations'] not in index"

In [85]:
# ==================================================
# 7.4B Preview NFIP-Enhanced Dataset
# ==================================================

# Check available disaster-related columns
print("Disaster-related columns:")
print([col for col in county_year_nfip.columns if "disaster" in col.lower()])

# Preview selected columns that definitely exist
preview_cols = [
    "STCOFIPS",
    "Year",
    "avg_annual_housing_price",
    "median_household_income",
    "price_to_income_ratio",
    "nfip_claim_count",
    "nfip_total_claim_payment",
    "nfip_cumulative_claim_payment",
    "nfip_recent_3yr_claim_payment",
    "nfip_claim_year_indicator"
]

display(county_year_nfip[preview_cols].head())

Disaster-related columns:
['disaster_count', 'unique_disaster_events', 'climate_disaster_count', 'hurricane_disaster_count', 'tropical_storm_disaster_count', 'severe_storm_disaster_count', 'flood_disaster_count', 'fire_disaster_count', 'any_disaster_flag', 'any_climate_disaster_flag', 'cumulative_disaster_count', 'cumulative_climate_disaster_count', 'recent_3yr_disaster_count', 'recent_3yr_climate_disaster_count']


,STCOFIPS,Year,avg_annual_housing_price,median_household_income,price_to_income_ratio,nfip_claim_count,nfip_total_claim_payment,nfip_cumulative_claim_payment,nfip_recent_3yr_claim_payment,nfip_claim_year_indicator
0,12001,2011,144811.606954,41373.0,3.500148,0.0,0.00,0.00,0.00,0
1,12001,2012,136832.426441,42818.0,3.195675,10.0,200195.98,200195.98,200195.98,1
2,12001,2013,140015.149500,42149.0,3.321909,3.0,5630.43,205826.41,205826.41,1
3,12001,2014,147354.521716,42045.0,3.504686,2.0,12849.49,218675.90,218675.90,1
4,12001,2015,153750.122064,43073.0,3.569524,6.0,133664.06,352339.96,152143.98,1


#### 7.4B Output Interpretation

The NFIP-enhanced dataset preview confirms that the insurance-loss indicators were successfully added to the ACS-enhanced county-year dataset.

The available disaster-related columns show that disaster history is already represented through annual, cumulative, recent three-year, and hazard-specific disaster indicators. The preview also confirms that NFIP claim indicators align correctly with the housing and affordability variables by county and year.

For county-years with no NFIP claims, the claim count, claim payment, cumulative payment, recent payment, and claim-year indicator are zero. For county-years with claim activity, the annual, cumulative, and recent three-year indicators update as expected. This confirms that the NFIP insurance-loss layer is correctly structured for later vulnerability scoring and modelling.

### 7.5 Validate and Save NFIP-Enhanced Dataset

After merging the NFIP claims indicators with the ACS-enhanced county-year dataset, this step performs final validation checks and saves the NFIP-enhanced dataset.

The validation confirms that the merge preserved the expected number of county-year observations, retained all 67 Florida counties, covered the full 2011–2025 study period, avoided duplicate county-year rows, and introduced no missing values in the NFIP feature columns.

In [86]:
# ==================================================
# 7.5 Validate and Save NFIP-Enhanced Dataset
# ==================================================

# --------------------------------------------------
# Basic dataset validation
# --------------------------------------------------

print("NFIP-enhanced dataset shape:")
print(county_year_nfip.shape)

print("\nUnique counties:")
print(county_year_nfip["STCOFIPS"].nunique())

print("\nYear range:")
print(
    county_year_nfip["Year"].min(),
    "to",
    county_year_nfip["Year"].max()
)

print("\nDuplicate county-year rows:")
duplicate_county_year_rows = county_year_nfip.duplicated(
    subset=["STCOFIPS", "Year"]
).sum()
print(duplicate_county_year_rows)

# --------------------------------------------------
# Validate expected county-year structure
# --------------------------------------------------

expected_rows = len(county_year_acs_projected)
actual_rows = len(county_year_nfip)

print("\nExpected rows:")
print(expected_rows)

print("\nActual rows:")
print(actual_rows)

print("\nRow count preserved:")
print(expected_rows == actual_rows)

expected_counties = county_year_acs_projected["STCOFIPS"].nunique()
actual_counties = county_year_nfip["STCOFIPS"].nunique()

print("\nExpected counties:")
print(expected_counties)

print("\nActual counties:")
print(actual_counties)

print("\nCounty count preserved:")
print(expected_counties == actual_counties)

# --------------------------------------------------
# Validate NFIP feature completeness
# --------------------------------------------------

nfip_feature_cols = [
    "nfip_claim_count",
    "nfip_policy_count_sum",
    "nfip_total_building_payment",
    "nfip_total_contents_payment",
    "nfip_total_icc_payment",
    "nfip_total_claim_payment",
    "nfip_avg_claim_payment",
    "nfip_total_building_coverage",
    "nfip_total_contents_coverage",
    "nfip_cumulative_claim_count",
    "nfip_cumulative_claim_payment",
    "nfip_recent_3yr_claim_count",
    "nfip_recent_3yr_claim_payment",
    "nfip_claim_year_indicator"
]

print("\nMissing values in NFIP feature columns:")
print(county_year_nfip[nfip_feature_cols].isna().sum())

print("\nTotal missing NFIP feature values:")
print(county_year_nfip[nfip_feature_cols].isna().sum().sum())

# --------------------------------------------------
# Validate NFIP indicator ranges
# --------------------------------------------------

print("\nNFIP claim-year indicator values:")
print(county_year_nfip["nfip_claim_year_indicator"].value_counts().sort_index())

print("\nMinimum values for selected NFIP features:")
print(
    county_year_nfip[
        [
            "nfip_claim_count",
            "nfip_total_claim_payment",
            "nfip_cumulative_claim_count",
            "nfip_cumulative_claim_payment",
            "nfip_recent_3yr_claim_count",
            "nfip_recent_3yr_claim_payment"
        ]
    ].min()
)

print("\nMaximum values for selected NFIP features:")
print(
    county_year_nfip[
        [
            "nfip_claim_count",
            "nfip_total_claim_payment",
            "nfip_cumulative_claim_count",
            "nfip_cumulative_claim_payment",
            "nfip_recent_3yr_claim_count",
            "nfip_recent_3yr_claim_payment"
        ]
    ].max()
)

# --------------------------------------------------
# Create final validation summary table
# --------------------------------------------------

nfip_validation_summary = pd.DataFrame({
    "Check": [
        "Final dataset rows",
        "Final dataset columns",
        "Unique counties",
        "Start year",
        "End year",
        "Duplicate county-year rows",
        "Expected rows preserved",
        "Expected counties preserved",
        "Missing NFIP feature values",
        "County-years with NFIP claims",
        "County-years without NFIP claims"
    ],
    "Value": [
        county_year_nfip.shape[0],
        county_year_nfip.shape[1],
        county_year_nfip["STCOFIPS"].nunique(),
        county_year_nfip["Year"].min(),
        county_year_nfip["Year"].max(),
        duplicate_county_year_rows,
        expected_rows == actual_rows,
        expected_counties == actual_counties,
        county_year_nfip[nfip_feature_cols].isna().sum().sum(),
        int((county_year_nfip["nfip_claim_year_indicator"] == 1).sum()),
        int((county_year_nfip["nfip_claim_year_indicator"] == 0).sum())
    ]
})

display(nfip_validation_summary)

# --------------------------------------------------
# Save final NFIP-enhanced county-year dataset
# --------------------------------------------------

nfip_enhanced_path = (
    INTERIM_DATA
    / "county_year_housing_spatial_disaster_acs_nfip_features.csv"
)

county_year_nfip.to_csv(nfip_enhanced_path, index=False)

print("\nSaved NFIP-enhanced county-year dataset to:")
print(nfip_enhanced_path)

NFIP-enhanced dataset shape:
(1000, 75)

Unique counties:
67

Year range:
2011 to 2025

Duplicate county-year rows:
0

Expected rows:
1000

Actual rows:
1000

Row count preserved:
True

Expected counties:
67

Actual counties:
67

County count preserved:
True

Missing values in NFIP feature columns:
nfip_claim_count                 0
nfip_policy_count_sum            0
nfip_total_building_payment      0
nfip_total_contents_payment      0
nfip_total_icc_payment           0
nfip_total_claim_payment         0
nfip_avg_claim_payment           0
nfip_total_building_coverage     0
nfip_total_contents_coverage     0
nfip_cumulative_claim_count      0
nfip_cumulative_claim_payment    0
nfip_recent_3yr_claim_count      0
nfip_recent_3yr_claim_payment    0
nfip_claim_year_indicator        0
dtype: int64

Total missing NFIP feature values:
0

NFIP claim-year indicator values:
nfip_claim_year_indicator
0    241
1    759
Name: count, dtype: int64

Minimum values for selected NFIP features:
nfip_claim

,Check,Value
0,Final dataset rows,1000
1,Final dataset columns,75
2,Unique counties,67
3,Start year,2011
4,End year,2025
5,Duplicate county-year rows,0
6,Expected rows preserved,True
7,Expected counties preserved,True
8,Missing NFIP feature values,0
9,County-years with NFIP claims,759



Saved NFIP-enhanced county-year dataset to:
..\data\interim\county_year_features\county_year_housing_spatial_disaster_acs_nfip_features.csv


#### 7.5 Output Interpretation

The NFIP-enhanced county-year dataset was successfully validated and saved.

The final dataset contains 1,000 county-year observations, 75 columns, 67 Florida counties, and covers the full 2011–2025 study period. The merge preserved the expected row count and county count, introduced no duplicate county-year rows, and produced no missing values in the NFIP feature columns.

The NFIP claim-year indicator shows that 759 county-years had at least one NFIP claim, while 241 county-years had no NFIP claims. This confirms that the dataset captures both claim and no-claim county-year conditions.

The NFIP variables include annual claim activity, total claim payments, cumulative claim burden, and recent three-year claim burden. These indicators add an insurance-loss and flood-related financial stress layer to the climate-housing vulnerability dataset.

The validated dataset was saved as `county_year_housing_spatial_disaster_acs_nfip_features.csv` and will be used as the input for vulnerability score construction and later machine learning analysis.